# Maternal LFP Data Preprocessing — Overview

This notebook builds **all spectral feature `.pkl` files** consumed by the six downstream task notebooks (OnnestVsOffnest, LickingVsNonLicking, LickingVsGrooming, PreVsPost134; in both 3-band and 1Hz-band variants).

## Pipeline at a glance

| # | Section | What it does | Generates |
|---|---|---|---|
| 2 | **Spectral extraction (3-band)** | Welch power + coherence on 8 ROIs across 3 wide bands `(2-7), (8-12), (14-23)` Hz; combine per-mouse outputs | `Spec_Features_8Yes/combine/full_spec_features_8roi.pkl` |
| 3 | **Onnest labels (Apr 3)** | Read on-nest behavior Excel, generate per-window 0/1 labels; bind to LFP windows | per-mouse `_P{1,3,8}.pkl` (updated in place) |
| 6 | **Onnest merge (Sep 4)** | Concatenate per-mouse onnest features across P1/P3/P8 | `Spec_Features_8Yes/combined/full_onnest_spec_features_8roi.pkl` |
| 7 | **Licking + Selfgrooming + Nursing** | Add licking/selfgroom/nursing labels from Excel to per-mouse pkls; aggregate two main bundles | `Combined/P3_onnest_lick/full_onnest_lick.pkl`<br>`Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_{with_nursing,no_nursing_field}.pkl` |
| 8 | **Pup retrieval (Oct 22)** | Extract P4 home-cage pup-retrieval windows | `Combined/P4_pup_retrieval_detail.pkl` |
| 9 | **Trim LFP** | Drop noisy windows from each pkl; create the `_Trim` variants used by tasks | `..._Trim.pkl` × 4: stage / onnest / lick / selfgroom |
| 10 | **Sara's Jan 6 request** | Align nursing vs no-nursing datasets and trim to 3600 samples per mouse | `..._all_behaviors_complete_trimmed.pkl` |
| 11 | **Jan 12 onnest fix** | Repair E5ELS41 P1 onnest labels (video gap); re-aggregate as `Jan212026` series | `..._8roi_Jan212026_Trim.pkl` |
| 12 | **Jan 22 — 1Hz frequency band** | Re-run Welch with 1Hz-wide bands across 2-56 Hz (54 bins); produce 1Hz-resolution feature pkls | `Spec_Features_1Hz_8roi/combined/full_spec_features_Trim_All.pkl`<br>`Spec_Features_1Hz_8roi/combined/full_onnest_spec_features_Trim.pkl` |

## Datasets consumed by downstream task notebooks (10 pkls total)

**3-band variants** (in `Spec_Features_8Yes/`):
- `full_spec_features_8roi.pkl` — full stage data
- `full_spec_features_8roi_Trim_All.pkl` — trimmed stage data
- `full_onnest_spec_features_8roi_Trim.pkl` — onnest subset, trimmed
- `full_onnest_spec_features_8roi_Jan212026_Trim.pkl` — Jan-2026 corrected onnest subset

**1Hz-band variants** (in `Spec_Features_1Hz_8roi/`):
- `full_spec_features_Trim_All.pkl` — full stage, 1Hz resolution
- `full_onnest_spec_features_Trim.pkl` — onnest subset, 1Hz resolution

**Cross-task auxiliaries** (in `Spec_Features_All/Combined/`):
- `P4_pup_retrieval_detail.pkl` — pup retrieval windows
- `P3_onnest_lick/full_onnest_lick_Trim.pkl` — licking labels, trimmed
- `P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field_Trim.pkl` — selfgroom/lick/onnest combined
- `P3_onnest_lick_selfgroom_nurse/full_all_behaviors_complete_trimmed.pkl` — with nursing aligned

## Conventions
- **8 ROIs**: PrL, IL, NAc, BLA, CeA, MeA, VHipp, VTA
- **Window**: 3.0 s (3-band pipeline) / 1.0 s (1Hz pipeline)
- **Periods**: Pre, Ges, P1, P3, P4, P8, P14, P20 — only the subset present per recording is used
- **Cohorts**: 8 C-group mice (control) + 6 E-group mice (early-life stress)


# Environment Check

# Self-defined functions

In [2]:
def get_filenames(label_dir, label_suffix):
    assert os.path.exists(label_dir), f"{label_dir} doesn't exist!"
    fns = [
        os.path.join(label_dir, fn)
        for fn in sorted(os.listdir(label_dir))
        if fn.endswith(label_suffix) and not fn.startswith("~$")  # filter out tempor files
    ]
    if len(fns) == 0:
        warnings.warn(f"No files in {label_dir}!")
    else: print(f"{len(fns)} files in {label_dir}")
    return fns

In [22]:
import pandas as pd
import numpy as np

# Load the Excel file
file_path = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Sara's Histology.xlsx"  # Update this path
df = pd.read_excel(file_path, sheet_name='Animal List by Region', skiprows=1)

print("Raw data loaded:")
print("Columns:", df.columns.tolist())

# Set Animal ID as index
df = df.set_index(df.columns[0])  # Use first column as index
df.index.name = 'Animal ID'

# Remove any NaN rows
df = df.dropna(how='all')

# Keep only the 8 brain region columns (note: it's 'Vhipp' not 'VHipp')
region_columns = ['PrL', 'IL', 'Nac', 'MeA', 'CeA', 'BLA', 'Vhipp', 'VTA']
df = df[region_columns]

print(f"\nAfter cleaning:")
print(f"Index name: {df.index.name}")
print(f"Columns: {df.columns.tolist()}")
print(f"Shape: {df.shape}")
print("\nSample data:")
print(df.head())

# Filter 1: All regions YES
print("\n" + "="*60)
print("FILTER 1: Animals with ALL 8 regions = YES")
print("="*60)

all_yes_mask = df.eq('YES').all(axis=1)
all_yes_animals = df[all_yes_mask]

print(f"Animals with ALL 8 regions YES: {len(all_yes_animals)}")
if len(all_yes_animals) > 0:
    print("\nAnimal IDs:")
    for animal_id in all_yes_animals.index:
        print(f"  - {animal_id}")
    
    print(f"\nDetailed view:")
    print(all_yes_animals)
else:
    print("No animals found with ALL 8 regions = YES")

# Filter 2: All regions YES except BLA and CeA  
print("\n" + "="*60)
print("FILTER 2: Animals with ALL 6 regions YES (excluding BLA and CeA)")
print("="*60)

non_amygdala_regions = ['PrL', 'IL', 'Nac', 'MeA', 'Vhipp', 'VTA']
print(f"Checking these 6 regions: {non_amygdala_regions}")
print("(BLA and CeA can be YES or NO - we don't care)")

all_except_amygdala_mask = df[non_amygdala_regions].eq('YES').all(axis=1)
all_except_amygdala_animals = df[all_except_amygdala_mask]

print(f"Animals with 6 regions YES (ignoring BLA, CeA): {len(all_except_amygdala_animals)}")
if len(all_except_amygdala_animals) > 0:
    print("\nAnimal IDs:")
    for animal_id in all_except_amygdala_animals.index:
        print(f"  - {animal_id}")
    
    print(f"\nDetailed view (showing all 8 columns):")
    print(all_except_amygdala_animals)
    
    # Show breakdown of BLA and CeA status for these animals
    print(f"\nBLA and CeA status for these {len(all_except_amygdala_animals)} animals:")
    for animal_id in all_except_amygdala_animals.index:
        bla_status = all_except_amygdala_animals.loc[animal_id, 'BLA']
        cea_status = all_except_amygdala_animals.loc[animal_id, 'CeA']
        print(f"  {animal_id}: BLA={bla_status}, CeA={cea_status}")
else:
    print("No animals found with 6 regions YES (excluding BLA, CeA)")

# Show the difference between the two filters
print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"Filter 1 (all 8 regions YES): {len(all_yes_animals)} animals")
print(f"Filter 2 (6 regions YES, ignoring BLA/CeA): {len(all_except_amygdala_animals)} animals")
print(f"Difference: {len(all_except_amygdala_animals) - len(all_yes_animals)} additional animals in Filter 2")

# Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print("Region coverage summary:")
for region in region_columns:
    yes_count = (df[region] == 'YES').sum()
    no_count = (df[region] == 'NO').sum()
    total = len(df)
    print(f"  {region:<8}: {yes_count:3d} YES, {no_count:3d} NO, {yes_count/total*100:5.1f}% coverage")

Raw data loaded:
Columns: ['Animal ID', 'PrL', 'IL', 'Nac', 'MeA', 'CeA', 'BLA', 'Vhipp', 'VTA', 'Unnamed: 9']

After cleaning:
Index name: Animal ID
Columns: ['PrL', 'IL', 'Nac', 'MeA', 'CeA', 'BLA', 'Vhipp', 'VTA']
Shape: (43, 8)

Sample data:
           PrL   IL  Nac  MeA  CeA  BLA Vhipp  VTA
Animal ID                                         
C1_ELS1    YES  YES  YES  YES  YES  YES   YES  YES
C6_ELS9    YES  YES  YES  YES  YES  YES   YES  YES
C7_ELS11   YES  YES  YES  YES  YES  YES   YES  YES
E2_ELS3    YES  YES  YES  YES  YES  YES   YES  YES
E5_ELS8    YES   NO  YES  YES  YES  YES   YES  YES

FILTER 1: Animals with ALL 8 regions = YES
Animals with ALL 8 regions YES: 18

Animal IDs:
  - C1_ELS1
  - C6_ELS9
  - C7_ELS11
  - E2_ELS3
  - C2_ELS18
  - C5_ELS20
  - C7_ELS22
  - C1_ELS32
  - C5_ELS40
  - C6_ELS42
  - C7_ELS45
  - E1_ELS33
  - E2_ELS35
  - E3_ELS37
  - E4_ELS39
  - E5_ELS41
  - E6_ELS43
  - E6_ELS44

Detailed view:
           PrL   IL  Nac  MeA  CeA  BLA Vhipp  VTA
Animal 

In [48]:
all_files_8roi = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes", ".pkl")

108 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes


In [49]:
# Initialize data structure to hold all combined data
all_data = {
    'power': [],
    'coh_sq_coherence': [],
    'freq_band': None,
    'region': None,
    'region_pair': None,
    'mouse_id': [],
    'period': []
}

# Use all_files_8roi as the source of files to process
files_to_process = all_files_8roi

# Track files that fail to load
failed_files = []

# Load and combine data from each file
print("Loading and combining feature files...")
print(f"Processing {len(files_to_process)} files from Spec_Features_8Yes directory")

for i, file_path in enumerate(files_to_process):
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
            
            # Append array data
            all_data['power'].append(data['power'])
            all_data['coh_sq_coherence'].append(data['coh_sq_coherence'])
            all_data['mouse_id'].append(data['mouse_id'])
            all_data['period'].append(data['period'])
            
            # Store static data (should be the same across all files)
            if all_data['freq_band'] is None:
                all_data['freq_band'] = data['freq_band']
            if all_data['region'] is None:
                all_data['region'] = data['region']
            if all_data['region_pair'] is None:
                all_data['region_pair'] = data['region_pair']
                
        if (i+1) % 50 == 0:
            print(f"  Processed {i+1}/{len(files_to_process)} files")
            
    except Exception as e:
        print(f"  ERROR loading file {file_path}: {str(e)}")
        failed_files.append((file_path, str(e)))

# Concatenate all the array data
print("\nConcatenating arrays...")
all_data['power'] = np.concatenate(all_data['power'], axis=0)
all_data['coh_sq_coherence'] = np.concatenate(all_data['coh_sq_coherence'], axis=0)
all_data['mouse_id'] = np.concatenate(all_data['mouse_id'], axis=0)
all_data['period'] = np.concatenate(all_data['period'], axis=0)

# Save the combined data - create combine folder in the specified directory
output_dir = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "full_spec_features_8roi.pkl")
print(f"Saving combined data to {output_path}...")
with open(output_path, 'wb') as file:
    pickle.dump(all_data, file)

# Print summary
print("\n" + "-" * 80)
print(f"COMBINATION COMPLETE")
print(f"  Total files processed: {len(files_to_process) - len(failed_files)}/{len(files_to_process)}")
print(f"  Failed files: {len(failed_files)}")
print(f"  Output file: {output_path}")
print(f"  Data shape summary:")
print(f"    - power: {all_data['power'].shape}")
print(f"    - coh_sq_coherence: {all_data['coh_sq_coherence'].shape}")
print(f"    - mouse_id: {all_data['mouse_id'].shape}")
if 'period' in all_data and len(all_data['period']) > 0:
    print(f"    - period: {all_data['period'].shape}")
print("-" * 80)

if failed_files:
    print("\nList of failed files:")
    for i, (file, error) in enumerate(failed_files, 1):
        print(f"  {i}. {file} - Error: {error}")
        
# Optional: Show some statistics about the regions
if all_data['region'] is not None:
    print(f"\nRegions in the combined data: {all_data['region']}")
    print(f"Number of regions: {len(all_data['region'])}")

if all_data['region_pair'] is not None:
    print(f"Number of region pairs: {len(all_data['region_pair'])}")

Loading and combining feature files...
Processing 108 files from Spec_Features_8Yes directory
  Processed 50/108 files
  Processed 100/108 files

Concatenating arrays...
Saving combined data to /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi.pkl...

--------------------------------------------------------------------------------
COMBINATION COMPLETE
  Total files processed: 108/108
  Failed files: 0
  Output file: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi.pkl
  Data shape summary:
    - power: (190145, 24)
    - coh_sq_coherence: (190145, 84)
    - mouse_id: (190145,)
    - period: (190145,)
--------------------------------------------------------------------------------

Regions in the combined data: ['BLA', 'CeA', 'IL', 'MeA', 'NAc', 'PrL', 'VHipp', 'VTA', 'BLA', 'CeA', 'IL', 'MeA', 'NAc', 'PrL', 'VHipp', 'VTA', 'BLA', 'CeA', 'IL', 'MeA', 'NAc', 'PrL', 'VH

# Complete 8 region and 6 region reorg - end

# 3. Apr 3 2025 Onnest Process -- P1/3/8 add onnest labels to each individual -- before histology

+ full onnest_label (Before histology) (P1+P3+P8): "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/full_onnest_spec_features.pkl"
+ individual spectral feature pkl is updated with onnest label -- 'onnest_label'
+ '/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/MouseE5F1ELS41_P1.pkl' was intentionally removed waiting for Sara to update onnest excel at that time

In [67]:
def generate_labels1(n_window, window_duration, onnest_data):
    # n_window0
    labels = np.zeros(n_window, dtype=int)
    current_nest_index = 0
    
    # - 'start' 'stop'
    start_col = None
    stop_col = None
    
    for col in onnest_data.columns:
        if col.upper() == 'START':
            start_col = col
        elif col.upper() == 'STOP':
            stop_col = col
    
    if start_col is None or stop_col is None:
        raise ValueError("cannot find start or stop column")
    
    for i in range(n_window):
        window_start = i * window_duration
        window_end = (i + 1) * window_duration
        
        in_nest_time = 0
        while current_nest_index < len(onnest_data):
            start = onnest_data.iloc[current_nest_index][start_col]
            stop = onnest_data.iloc[current_nest_index][stop_col]
            
            if stop <= window_start:
                current_nest_index += 1
                continue
            
            if start >= window_end:
                break
            
            overlap_start = max(window_start, start)
            overlap_end = min(window_end, stop)
            
            if overlap_start < overlap_end:
                in_nest_time += (overlap_end - overlap_start)
            
            if stop <= window_end:
                current_nest_index += 1
            else:
                break
        
        # 1
        if in_nest_time >= (window_duration / 2):
            labels[i] = 1
    
    return {'onnest_label': labels}


In [68]:
all_fns = get_filenames('/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All', 'pkl')

246 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All


In [69]:
# Group files by stage using string methods
ges_fns = [fn for fn in all_fns if fn.endswith('_Ges.pkl')]
p1_fns = [fn for fn in all_fns if fn.endswith('_P1.pkl')]
p3_fns = [fn for fn in all_fns if fn.endswith('_P3.pkl')]
p8_fns = [fn for fn in all_fns if fn.endswith('_P8.pkl')]
p14_fns = [fn for fn in all_fns if fn.endswith('_P14.pkl')]
p20_fns = [fn for fn in all_fns if fn.endswith('_P20.pkl')]
p4_home_fns = [fn for fn in all_fns if fn.endswith('_P4 home.pkl')]
p4_open_fns = [fn for fn in all_fns if fn.endswith('_P4 open.pkl')]
pre_home_fns = [fn for fn in all_fns if fn.endswith('_Pre home.pkl')]
pre_pup_fns = [fn for fn in all_fns if fn.endswith('_Pre pup.pkl')]

# Print count for each stage
print(f"Ges: {len(ges_fns)} files")
print(f"P1: {len(p1_fns)} files")
print(f"P3: {len(p3_fns)} files")
print(f"P8: {len(p8_fns)} files")
print(f"P14: {len(p14_fns)} files")
print(f"P20: {len(p20_fns)} files")
print(f"P4 home: {len(p4_home_fns)} files")
print(f"P4 open: {len(p4_open_fns)} files")
print(f"Pre home: {len(pre_home_fns)} files")
print(f"Pre pup: {len(pre_pup_fns)} files")

# Calculate total files across all stages
total_stage_files = (len(ges_fns) + len(p1_fns) + len(p3_fns) + len(p8_fns) + 
                     len(p14_fns) + len(p20_fns) + len(p4_home_fns) + 
                     len(p4_open_fns) + len(pre_home_fns) + len(pre_pup_fns))

# Verify that all files are accounted for
print(f"\nTotal files across all stages: {total_stage_files}")
print(f"Total files in all_fns: {len(all_fns)}")

Ges: 32 files
P1: 24 files
P3: 30 files
P8: 31 files
P14: 27 files
P20: 26 files
P4 home: 10 files
P4 open: 10 files
Pre home: 42 files
Pre pup: 14 files

Total files across all stages: 246
Total files in all_fns: 246


In [70]:
P3_onnest_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/on-nest/Excel",
                              "xlsx")
P1_onnest_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P1 Recordings/Behavioral Scoring/on-nest",
                              "xlsx")
P8_onnest_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P8 Recordings/Behavioral Scoring/on-nest",
                              "xlsx")

34 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/on-nest/Excel
25 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P1 Recordings/Behavioral Scoring/on-nest
33 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P8 Recordings/Behavioral Scoring/on-nest


=== P3 mismatched mice ===
['MouseC1ELS1', 'MouseC6ELS9', 'MouseE1ELS2', 'MouseE8ELS14']  -------- 34 - 4 = 30 matched
=== P8 mismatched mice === 
['MouseC6ELS9', 'MouseE1ELS2']  -------- 33 - 2 = 31 matched

List of skipped files:
  1. P1/MouseE3F1ELS37_241014_001_LFP.mat  -------- 25 - 1 = 24 matched

+ P1/MouseE3F1ELS37_241014_001_LFP.mat does not exist as power becomes negative after log transformation

In [38]:
# Create a dictionary for quick lookup of P8 onnest files
onnest_dict = {}
for fn in P8_onnest_fns:
    match = re.search(r'/([CE]\d+)_ELS(\d+)\.xlsx$', fn)
    if match:
        cage, els = match.group(1), match.group(2)
        onnest_dict[(cage, els)] = fn

# Process matching files
matches = []
no_matches = []
window_duration = 3.0  # Default value

for p8_fn in p8_fns:
    match = re.search(r'Mouse([CE]\d+)(?:F\d+)?ELS(\d+)_P8\.pkl$', p8_fn)
    if not match:
        no_matches.append(p8_fn)
        continue
        
    cage, els = match.group(1), match.group(2)
    
    if (cage, els) in onnest_dict:
        onnest_fn = onnest_dict[(cage, els)]
        
        try:
            # Read data
            with open(p8_fn, 'rb') as file:
                feature_data = pickle.load(file)
            
            onnest_data = pd.read_excel(onnest_fn)
            
            # Calculate and update labels
            n_window = (feature_data['power'].shape)[0]
            onnest_label = generate_labels1(n_window, window_duration, onnest_data)
            feature_data.update(onnest_label)
            
            # Save back to original file
            with open(p8_fn, 'wb') as file:
                pickle.dump(feature_data, file)
            
            matches.append((p8_fn, onnest_fn))
        except Exception as e:
            print(f"Error processing {p8_fn}: {e}")
    else:
        no_matches.append(p8_fn)

print(f"Successfully processed {len(matches)} P8 files")
print(f"Files without matches: {len(no_matches)}")

Successfully processed 31 P8 files
Files without matches: 0


### Generate 2 types of onnest labels

In [ ]:
def generate_labels_with_duration(n_window, window_duration, onnest_data, bout_cutoff=60.0):
    """
    on-neston-nest
    
    :
    - n_window: 
    - window_duration: 
    - onnest_data: startstopDataFrame
    - bout_cutoff: on-nest10.0
    
    :
    - 0=off-nest1=short on-nest2=long on-nest
    """
    # n_window0
    labels = np.zeros(n_window, dtype=int)
    
    # - 'start' 'stop'
    start_col = None
    stop_col = None
    
    for col in onnest_data.columns:
        if col.lower().find('start') != -1:
            start_col = col
        elif col.lower().find('stop') != -1:
            stop_col = col
    
    if start_col is None or stop_col is None:
        raise ValueError("Cannot find start or stop column")
    
    valid_bouts = []
    invalid_count = 0
    
    sorted_data = onnest_data.sort_values(by=start_col).reset_index(drop=True)
    prev_stop = -1
    
    for i in range(len(sorted_data)):
        start = sorted_data.iloc[i][start_col]
        stop = sorted_data.iloc[i][stop_col]
        
        if stop < start:
            invalid_count += 1
            print(f"Warning: Removed invalid bout with negative duration: start={start:.2f}, stop={stop:.2f}")
            continue
            
        if start < prev_stop:
            invalid_count += 1
            print(f"Warning: Removed overlapping bout: start={start:.2f}, previous stop={prev_stop:.2f}")
            continue
            
        duration = stop - start
        is_long = duration > bout_cutoff
        valid_bouts.append({
            'start': start,
            'stop': stop,
            'duration': duration,
            'is_long': is_long
        })
        prev_stop = stop
        
    if invalid_count > 0:
        print(f"Total {invalid_count} invalid bouts removed")
    
    print(f"Processing {len(valid_bouts)} valid bouts (cutoff={bout_cutoff}s)")
    
    for i in range(n_window):
        window_start = i * window_duration
        window_end = (i + 1) * window_duration
        
        off_nest_time = window_duration  # off-nest
        short_on_nest_time = 0
        long_on_nest_time = 0
        
        for bout in valid_bouts:
            if bout['stop'] <= window_start:
                continue
                
            if bout['start'] >= window_end:
                break
                
            overlap_start = max(window_start, bout['start'])
            overlap_end = min(window_end, bout['stop'])
            overlap_duration = overlap_end - overlap_start
            
            if overlap_duration > 0:
                off_nest_time -= overlap_duration
                if bout['is_long']:
                    long_on_nest_time += overlap_duration
                else:
                    short_on_nest_time += overlap_duration
        
        max_time = max(off_nest_time, short_on_nest_time, long_on_nest_time)
        
        if max_time == off_nest_time:
            labels[i] = 0  # off-nest
        elif max_time == short_on_nest_time:
            labels[i] = 1  # short on-nest
        else:
            labels[i] = 2  # long on-nest
    
    label_counts = {
        0: np.sum(labels == 0),
        1: np.sum(labels == 1),
        2: np.sum(labels == 2)
    }
    print(f"Generated labels: {label_counts[0]} off-nest, {label_counts[1]} short on-nest, {label_counts[2]} long on-nest")
    
    return {'onnest_label': labels}


In [ ]:
all_fns = get_filenames('/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All', 'pkl')

p1_fns = [fn for fn in all_fns if fn.endswith('_P1.pkl')]
p3_fns = [fn for fn in all_fns if fn.endswith('_P3.pkl')]
p8_fns = [fn for fn in all_fns if fn.endswith('_P8.pkl')]

P3_onnest_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/on-nest/Excel",
                              "xlsx")
P1_onnest_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P1 Recordings/Behavioral Scoring/on-nest",
                              "xlsx")
P8_onnest_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P8 Recordings/Behavioral Scoring/on-nest",
                              "xlsx")

# Create a dictionary for quick lookup of onnest files
onnest_dict = {}
for fn in P3_onnest_fns:
    match = re.search(r'/([CE]\d+)_ELS(\d+)\.xlsx$', fn)
    if match:
        cage, els = match.group(1), match.group(2)
        onnest_dict[(cage, els)] = fn

# Process matching files
matches = []
no_matches = []
window_duration = 3.0  # Default value

for p3_fn in p3_fns:
    match = re.search(r'Mouse([CE]\d+)(?:F\d+)?ELS(\d+)_P3\.pkl$', p3_fn)
    if not match:
        no_matches.append(p3_fn)
        continue
        
    cage, els = match.group(1), match.group(2)
    
    if (cage, els) in onnest_dict:
        onnest_fn = onnest_dict[(cage, els)]
        
        try:
            # Read data
            with open(p3_fn, 'rb') as file:
                feature_data = pickle.load(file)
            
            onnest_data = pd.read_excel(onnest_fn)
            
            # Calculate and update labels
            n_window = (feature_data['power'].shape)[0]
            label_dict = generate_labels_with_duration(n_window, window_duration, onnest_data, bout_cutoff = 60.0)
            feature_data['onnest_label2'] = label_dict['onnest_label']
            
            # Save back to original file
            with open(p3_fn, 'wb') as file:
                pickle.dump(feature_data, file)
            
            matches.append((p3_fn, onnest_fn))
        except Exception as e:
            print(f"Error processing {p3_fn}: {e}")
    else:
        no_matches.append(p3_fn)

print(f"Successfully processed {len(matches)} files")

# Create a dictionary for quick lookup of P1 onnest files
onnest_dict = {}
for fn in P1_onnest_fns:
    match = re.search(r'/([CE]\d+)_ELS(\d+)\.xlsx$', fn)
    if match:
        cage, els = match.group(1), match.group(2)
        onnest_dict[(cage, els)] = fn

# Process matching files
matches = []
no_matches = []
window_duration = 3.0  # Default value

for p1_fn in p1_fns:
    match = re.search(r'Mouse([CE]\d+)(?:F\d+)?ELS(\d+)_P1\.pkl$', p1_fn)
    if not match:
        no_matches.append(p1_fn)
        continue
        
    cage, els = match.group(1), match.group(2)
    
    if (cage, els) in onnest_dict:
        onnest_fn = onnest_dict[(cage, els)]
        
        try:
            # Read data
            with open(p1_fn, 'rb') as file:
                feature_data = pickle.load(file)
            
            onnest_data = pd.read_excel(onnest_fn)
            
            # Calculate and update labels
            n_window = (feature_data['power'].shape)[0]
            label_dict = generate_labels_with_duration(n_window, window_duration, onnest_data, bout_cutoff = 60.0)
            feature_data['onnest_label2'] = label_dict['onnest_label']
            
            # Save back to original file
            with open(p1_fn, 'wb') as file:
                pickle.dump(feature_data, file)
            
            matches.append((p1_fn, onnest_fn))
        except Exception as e:
            print(f"Error processing {p1_fn}: {e}")
    else:
        no_matches.append(p1_fn)

print(f"Successfully processed {len(matches)} P1 files")
print(f"Files without matches: {len(no_matches)}")

# Create a dictionary for quick lookup of P8 onnest files
onnest_dict = {}
for fn in P8_onnest_fns:
    match = re.search(r'/([CE]\d+)_ELS(\d+)\.xlsx$', fn)
    if match:
        cage, els = match.group(1), match.group(2)
        onnest_dict[(cage, els)] = fn

# Process matching files
matches = []
no_matches = []
window_duration = 3.0  # Default value

for p8_fn in p8_fns:
    match = re.search(r'Mouse([CE]\d+)(?:F\d+)?ELS(\d+)_P8\.pkl$', p8_fn)
    if not match:
        no_matches.append(p8_fn)
        continue
        
    cage, els = match.group(1), match.group(2)
    
    if (cage, els) in onnest_dict:
        onnest_fn = onnest_dict[(cage, els)]
        
        try:
            # Read data
            with open(p8_fn, 'rb') as file:
                feature_data = pickle.load(file)
            
            onnest_data = pd.read_excel(onnest_fn)
            
            # Calculate and update labels
            n_window = (feature_data['power'].shape)[0]
            label_dict = generate_labels_with_duration(n_window, window_duration, onnest_data, bout_cutoff = 60.0)
            feature_data['onnest_label2'] = label_dict['onnest_label']
            
            # Save back to original file
            with open(p8_fn, 'wb') as file:
                pickle.dump(feature_data, file)
            
            matches.append((p8_fn, onnest_fn))
        except Exception as e:
            print(f"Error processing {p8_fn}: {e}")
    else:
        no_matches.append(p8_fn)

print(f"Successfully processed {len(matches)} P8 files")
print(f"Files without matches: {len(no_matches)}")


In [ ]:
Onnest_fns = p1_fns + p3_fns + p8_fns 

In [ ]:
all_onnest_data = {
    'power': [],
    'coh_sq_coherence': [],
    'freq_band': None,
    'region': None,
    'region_pair': None,
    'mouse_id': [],
    'period': [],
    'onnest_label': [],
    'onnest_label2': []  # add new onnest_label2: 0-offnest, 1-shortonnest, 2-longonnest (1mins)
}

# Load and combine data from each file
print("Loading and combining onnest feature files...")
for i, file_path in enumerate(Onnest_fns):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
        
        # Append array data
        all_onnest_data['power'].append(data['power'])
        all_onnest_data['coh_sq_coherence'].append(data['coh_sq_coherence'])
        all_onnest_data['mouse_id'].append(data['mouse_id'])
        all_onnest_data['period'].append(data['period'])
        all_onnest_data['onnest_label'].append(data['onnest_label'])
        all_onnest_data['onnest_label2'].append(data['onnest_label2']) 
        
        # Store static data from first file
        if i == 0:
            all_onnest_data['freq_band'] = data['freq_band']
            all_onnest_data['region'] = data['region']
            all_onnest_data['region_pair'] = data['region_pair']
            
    if (i+1) % 10 == 0:
        print(f"  Processed {i+1}/{len(Onnest_fns)} files")

# Concatenate all the array data
print("\nConcatenating arrays...")
all_onnest_data['power'] = np.concatenate(all_onnest_data['power'], axis=0)
all_onnest_data['coh_sq_coherence'] = np.concatenate(all_onnest_data['coh_sq_coherence'], axis=0)
all_onnest_data['mouse_id'] = np.concatenate(all_onnest_data['mouse_id'], axis=0)
all_onnest_data['period'] = np.concatenate(all_onnest_data['period'], axis=0)
all_onnest_data['onnest_label'] = np.concatenate(all_onnest_data['onnest_label'], axis=0)
all_onnest_data['onnest_label2'] = np.concatenate(all_onnest_data['onnest_label2'], axis=0)

# Save the combined data
output_path = os.path.join("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined", "full_onnest_spec_features.pkl")
print(f"Saving combined onnest data to {output_path}...")
with open(output_path, 'wb') as file:
    pickle.dump(all_onnest_data, file)

# Print summary
print("\n" + "-" * 80)
print(f"ONNEST COMBINATION COMPLETE")
print(f"  Total files processed: {len(Onnest_fns)}")
print(f"  Output file: {output_path}")
print(f"  Data shape summary:")
print(f"    - power: {all_onnest_data['power'].shape}")
print(f"    - coh_sq_coherence: {all_onnest_data['coh_sq_coherence'].shape}")
print(f"    - mouse_id: {all_onnest_data['mouse_id'].shape}")
print(f"    - period: {all_onnest_data['period'].shape}")
print(f"    - onnest_label: {all_onnest_data['onnest_label'].shape}")
print(f"    - onnest_label2: {all_onnest_data['onnest_label2'].shape}")

# 
label_counts = {
    0: np.sum(all_onnest_data['onnest_label2'] == 0),  
    1: np.sum(all_onnest_data['onnest_label2'] == 1),  
    2: np.sum(all_onnest_data['onnest_label2'] == 2)   
}
print(f"    - onnest_label2 distribution: {label_counts[0]} off-nest, {label_counts[1]} short on-nest, {label_counts[2]} long on-nest")
print("-" * 80)

# 6. Start Sep4 - Just pick up and merge onnest data for histology - identical to the previous generated

+ redo it b/c didn't realized the previous one has both spectral features and onnest labels (thought only spectral features)

In [ ]:
import numpy as np
import pickle
import re
import os

# Define target mice
target_mice = {
    'C1_ELS1', 'C6_ELS9', 'C7_ELS11', 'E2_ELS3', 'C2_ELS18', 
    'C5_ELS20', 'C7_ELS22', 'C1_ELS32', 'C5_ELS40', 'C6_ELS42', 
    'C7_ELS45', 'E1_ELS33', 'E2_ELS35', 'E3_ELS37', 'E4_ELS39', 
    'E5_ELS41', 'E6_ELS43', 'E6_ELS44'
}

# Function to convert mouse ID format
def convert_mouse_id(original_id):
    """
    Convert mouse ID to CX_ELSXX or EX_ELSXX format
    """
    # Pattern 1: MouseCXFXELSXX -> CX_ELSXX
    pattern1 = r'MouseC(\d+)F\d+ELS(\d+)'
    match1 = re.match(pattern1, original_id)
    if match1:
        return f"C{match1.group(1)}_ELS{match1.group(2)}"
    
    # Pattern 2: MouseCXELSXX -> CX_ELSXX
    pattern2 = r'MouseC(\d+)ELS(\d+)'
    match2 = re.match(pattern2, original_id)
    if match2:
        return f"C{match2.group(1)}_ELS{match2.group(2)}"
    
    # Pattern 3: MouseEXFXELSXX -> EX_ELSXX
    pattern3 = r'MouseE(\d+)F\d+ELS(\d+)'
    match3 = re.match(pattern3, original_id)
    if match3:
        return f"E{match3.group(1)}_ELS{match3.group(2)}"
    
    # Pattern 4: MouseEXELSXX -> EX_ELSXX
    pattern4 = r'MouseE(\d+)ELS(\d+)'
    match4 = re.match(pattern4, original_id)
    if match4:
        return f"E{match4.group(1)}_ELS{match4.group(2)}"
    
    # Pattern 5: Already in correct format CXELSXX -> CX_ELSXX
    pattern5 = r'C(\d+)ELS(\d+)'
    match5 = re.match(pattern5, original_id)
    if match5:
        return f"C{match5.group(1)}_ELS{match5.group(2)}"
    
    # Pattern 6: Already in correct format EXELSXX -> EX_ELSXX
    pattern6 = r'E(\d+)ELS(\d+)'
    match6 = re.match(pattern6, original_id)
    if match6:
        return f"E{match6.group(1)}_ELS{match6.group(2)}"
    
    return original_id  # Return original if no pattern matches

# Convert all mouse IDs
print("Converting mouse IDs...")
converted_mouse_ids = np.array([convert_mouse_id(mid) for mid in data['mouse_id']])

# Check unique converted IDs
unique_converted = np.unique(converted_mouse_ids)
print(f"Number of unique converted mouse IDs: {len(unique_converted)}")
print(f"Converted mouse IDs: {unique_converted}")

# Find intersection with target mice
intersection = set(unique_converted) & target_mice
print(f"\nIntersection with target mice: {intersection}")
print(f"Number of mice in intersection: {len(intersection)}")

# Create mask for filtering
mask = np.isin(converted_mouse_ids, list(intersection))
print(f"Number of samples matching target mice: {np.sum(mask)}")

# Filter all data based on the mask
filtered_data = {}
for key in data.keys():
    if key in ['mouse_id', 'period', 'onnest_label', 'onnest_label2']:
        # For 1D arrays, filter directly
        filtered_data[key] = data[key][mask]
    elif key in ['power', 'coh_sq_coherence']:
        # For 2D arrays, filter rows
        filtered_data[key] = data[key][mask]
    else:
        # For other keys (freq_band, region, region_pair), keep as is
        filtered_data[key] = data[key]

# Update mouse_id with converted format
filtered_data['mouse_id'] = converted_mouse_ids[mask]

# Print summary of filtered data
print(f"\nFiltered data summary:")
for key in filtered_data.keys():
    if hasattr(filtered_data[key], 'shape'):
        print(f"{key}: shape {filtered_data[key].shape}")
    else:
        print(f"{key}: {type(filtered_data[key])}, length {len(filtered_data[key])}")

# Check unique values in filtered data
print(f"\nUnique mouse IDs in filtered data: {np.unique(filtered_data['mouse_id'])}")

# Save filtered data
import os
file_path = '/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi.pkl'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'wb') as f:
    pickle.dump(filtered_data, f)

print(f"\nFiltered data saved to: {file_path}")

## end Sep4

# 7. May 15 -- licking Data -- Sep17 -- Oct1(selfgroom) -- P3 only

### visualization and check existance

In [21]:
P3_onnest_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/on-nest/Excel",
                              "xlsx")

P3_nurse_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/nursing",
                              "xlsx")

P3_lick_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/licking_grooming (of pups)",
                              "xlsx")

P3_selfgroom_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/non-maternal behaviors/grooming (of self)",
                              "xlsx")

34 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/on-nest/Excel
31 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/nursing
34 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/licking_grooming (of pups)
34 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/non-maternal behaviors/grooming (of self)


In [10]:
import os
import shutil
import re
from pathlib import Path

# Target Animal IDs from Filter 1 (all 8 regions YES)
target_animal_ids = [
    'C1_ELS1', 'C6_ELS9', 'C7_ELS11', 'E2_ELS3', 'C2_ELS18', 'C5_ELS20', 
    'C7_ELS22', 'C1_ELS32', 'C5_ELS40', 'C6_ELS42', 'C7_ELS45', 'E1_ELS33', 
    'E2_ELS35', 'E3_ELS37', 'E4_ELS39', 'E5_ELS41', 'E6_ELS43', 'E6_ELS44'
]

# Output directory
output_dir = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes"
os.makedirs(output_dir, exist_ok=True)

def extract_animal_id_from_filename(file_path):
    """
    Extract animal ID from file path, ignoring FXX part
    Examples:
    - MouseC1F3ELS32_Ges.pkl -> C1_ELS32
    - MouseE6F4ELS44_P1.pkl -> E6_ELS44
    - MouseC7ELS11_P3.pkl -> C7_ELS11
    """
    filename = Path(file_path).stem
    filename = filename.replace('Mouse', '')
    
    # Split by underscore to get animal part
    animal_part = filename.split('_')[0]
    
    # Extract animal ID using regex (ignore F part)
    match = re.match(r'([CE]\d+)(?:F\d+)?(ELS\d+)', animal_part)
    if match:
        prefix, els_part = match.groups()
        return f"{prefix}_{els_part}"
    return None

def should_include_file(file_path):
    """
    Check if file should be included (exclude Pre pup and P4 open)
    """
    filename = Path(file_path).stem
    
    # Exclude Pre pup and P4 open files
    if 'Pre pup' in filename or 'P4 open' in filename:
        return False
    return True

# Find and copy files for target animals
copied_files = []
skipped_files = []
excluded_files = []

print(f"Target animals: {len(target_animal_ids)}")
print(f"Total files to check: {len(all_fns)}")
print(f"Output directory: {output_dir}")
print("Excluding: Pre pup and P4 open files")
print("\nProcessing files...")

for file_path in all_fns:
    # Extract animal ID from filename
    animal_id = extract_animal_id_from_filename(file_path)
    
    if animal_id in target_animal_ids:
        # Check if file should be included (exclude Pre pup and P4 open)
        if should_include_file(file_path):
            # Copy file to output directory
            source_path = file_path
            filename = Path(file_path).name
            dest_path = os.path.join(output_dir, filename)
            
            try:
                shutil.copy2(source_path, dest_path)
                copied_files.append((animal_id, filename))
                print(f"✓ Copied: {filename} (Animal: {animal_id})")
            except Exception as e:
                print(f"✗ Error copying {filename}: {e}")
                skipped_files.append((animal_id, filename, str(e)))
        else:
            # File excluded due to stage filter
            filename = Path(file_path).name
            excluded_files.append((animal_id, filename))
            print(f"- Excluded: {filename} (Animal: {animal_id}) - Pre pup/P4 open")
    else:
        # File not in target list - skip
        continue

# Summary report
print(f"\n" + "="*80)
print("COPY OPERATION SUMMARY")
print("="*80)
print(f"Target animals: {len(target_animal_ids)}")
print(f"Files successfully copied: {len(copied_files)}")
print(f"Files excluded (Pre pup/P4 open): {len(excluded_files)}")
print(f"Files with errors: {len(skipped_files)}")

# Group by animal ID to show completeness
print(f"\nFiles copied per animal:")
from collections import defaultdict
files_per_animal = defaultdict(list)

for animal_id, filename in copied_files:
    files_per_animal[animal_id].append(filename)

for animal_id in sorted(target_animal_ids):
    if animal_id in files_per_animal:
        file_count = len(files_per_animal[animal_id])
        print(f"  {animal_id:<12}: {file_count:2d} files")
        # Show first few filenames as examples
        for i, filename in enumerate(sorted(files_per_animal[animal_id])[:3]):
            print(f"    - {filename}")
        if file_count > 3:
            print(f"    ... and {file_count - 3} more files")
    else:
        print(f"  {animal_id:<12}: 0 files (NOT FOUND)")

# Show excluded files summary
if excluded_files:
    print(f"\nExcluded files (Pre pup/P4 open):")
    excluded_per_animal = defaultdict(list)
    for animal_id, filename in excluded_files:
        excluded_per_animal[animal_id].append(filename)
    
    for animal_id in sorted(excluded_per_animal.keys()):
        file_count = len(excluded_per_animal[animal_id])
        print(f"  {animal_id}: {file_count} files excluded")

# Show any errors
if skipped_files:
    print(f"\nFiles with copy errors:")
    for animal_id, filename, error in skipped_files:
        print(f"  {animal_id} - {filename}: {error}")

# Verify output directory
output_files = os.listdir(output_dir)
print(f"\nOutput directory verification:")
print(f"  Directory: {output_dir}")
print(f"  Total files in output: {len(output_files)}")
print(f"  Expected files: {len(copied_files)}")

# Check if any target animals are completely missing
print(f"\nMissing animals check:")
animals_found = set(animal_id for animal_id, _ in copied_files)
missing_animals = set(target_animal_ids) - animals_found

if missing_animals:
    print(f"Animals with NO files found ({len(missing_animals)}):")
    for animal_id in sorted(missing_animals):
        print(f"  - {animal_id}")
else:
    print("✓ All target animals have at least one file")

print(f"\n✓ Copy operation completed!")
print(f"Files copied to: {output_dir}")

     mouse_id  onnest  nursing  licking  selfgroom
0   C10_ELS26       1        1        1          1
1   C11_ELS29       1        1        1          1
2     C1_ELS1       1        1        1          1
3    C1_ELS16       1        1        1          1
4    C1_ELS32       1        1        1          1
5    C2_ELS18       1        1        1          1
6    C3_ELS36       1        1        1          1
7    C4_ELS38       1        1        1          1
8    C5_ELS20       1        1        1          1
9    C5_ELS40       1        1        1          1
10   C6_ELS42       1        1        1          1
11    C6_ELS9       1        1        1          1
12   C7_ELS11       1        1        1          1
13   C7_ELS12       1        1        1          1
14   C7_ELS22       1        1        1          1
15   C7_ELS45       1        1        1          1
16   C8_ELS13       1        1        1          1
17   C9_ELS24       1        1        1          1
18  E11_ELS28       1        1 

### start to add response

+ Saving dataset WITH nursing to /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_with_nursing.pkl...
+ Saving dataset WITHOUT nursing field to /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field.pkl...

In [19]:
all_fns = get_filenames('/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All', 'pkl')
p3_fns = [fn for fn in all_fns if fn.endswith('_P3.pkl')]

246 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All


### add selfgrooming and nursing

In [12]:
import os
import shutil
import re
from pathlib import Path

# Target Animal IDs from Filter 1 (all 8 regions YES)
target_animal_ids = [
    'C1_ELS1', 'C6_ELS9', 'C7_ELS11', 'E2_ELS3', 'C2_ELS18', 'C5_ELS20', 
    'C7_ELS22', 'C1_ELS32', 'C5_ELS40', 'C6_ELS42', 'C7_ELS45', 'E1_ELS33', 
    'E2_ELS35', 'E3_ELS37', 'E4_ELS39', 'E5_ELS41', 'E6_ELS43', 'E6_ELS44'
]

# Output directory
output_dir = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes"
os.makedirs(output_dir, exist_ok=True)

def extract_animal_id_from_filename(file_path):
    """
    Extract animal ID from file path, ignoring FXX part
    Examples:
    - MouseC1F3ELS32_Ges.pkl -> C1_ELS32
    - MouseE6F4ELS44_P1.pkl -> E6_ELS44
    - MouseC7ELS11_P3.pkl -> C7_ELS11
    """
    filename = Path(file_path).stem
    filename = filename.replace('Mouse', '')
    
    # Split by underscore to get animal part
    animal_part = filename.split('_')[0]
    
    # Extract animal ID using regex (ignore F part)
    match = re.match(r'([CE]\d+)(?:F\d+)?(ELS\d+)', animal_part)
    if match:
        prefix, els_part = match.groups()
        return f"{prefix}_{els_part}"
    return None

def should_include_file(file_path):
    """
    Check if file should be included (exclude Pre pup and P4 open)
    """
    filename = Path(file_path).stem
    
    # Exclude Pre pup and P4 open files
    if 'Pre pup' in filename or 'P4 open' in filename:
        return False
    return True

# Find and copy files for target animals
copied_files = []
skipped_files = []
excluded_files = []

print(f"Target animals: {len(target_animal_ids)}")
print(f"Total files to check: {len(all_fns)}")
print(f"Output directory: {output_dir}")
print("Excluding: Pre pup and P4 open files")
print("\nProcessing files...")

for file_path in all_fns:
    # Extract animal ID from filename
    animal_id = extract_animal_id_from_filename(file_path)
    
    if animal_id in target_animal_ids:
        # Check if file should be included (exclude Pre pup and P4 open)
        if should_include_file(file_path):
            # Copy file to output directory
            source_path = file_path
            filename = Path(file_path).name
            dest_path = os.path.join(output_dir, filename)
            
            try:
                shutil.copy2(source_path, dest_path)
                copied_files.append((animal_id, filename))
                print(f"✓ Copied: {filename} (Animal: {animal_id})")
            except Exception as e:
                print(f"✗ Error copying {filename}: {e}")
                skipped_files.append((animal_id, filename, str(e)))
        else:
            # File excluded due to stage filter
            filename = Path(file_path).name
            excluded_files.append((animal_id, filename))
            print(f"- Excluded: {filename} (Animal: {animal_id}) - Pre pup/P4 open")
    else:
        # File not in target list - skip
        continue

# Summary report
print(f"\n" + "="*80)
print("COPY OPERATION SUMMARY")
print("="*80)
print(f"Target animals: {len(target_animal_ids)}")
print(f"Files successfully copied: {len(copied_files)}")
print(f"Files excluded (Pre pup/P4 open): {len(excluded_files)}")
print(f"Files with errors: {len(skipped_files)}")

# Group by animal ID to show completeness
print(f"\nFiles copied per animal:")
from collections import defaultdict
files_per_animal = defaultdict(list)

for animal_id, filename in copied_files:
    files_per_animal[animal_id].append(filename)

for animal_id in sorted(target_animal_ids):
    if animal_id in files_per_animal:
        file_count = len(files_per_animal[animal_id])
        print(f"  {animal_id:<12}: {file_count:2d} files")
        # Show first few filenames as examples
        for i, filename in enumerate(sorted(files_per_animal[animal_id])[:3]):
            print(f"    - {filename}")
        if file_count > 3:
            print(f"    ... and {file_count - 3} more files")
    else:
        print(f"  {animal_id:<12}: 0 files (NOT FOUND)")

# Show excluded files summary
if excluded_files:
    print(f"\nExcluded files (Pre pup/P4 open):")
    excluded_per_animal = defaultdict(list)
    for animal_id, filename in excluded_files:
        excluded_per_animal[animal_id].append(filename)
    
    for animal_id in sorted(excluded_per_animal.keys()):
        file_count = len(excluded_per_animal[animal_id])
        print(f"  {animal_id}: {file_count} files excluded")

# Show any errors
if skipped_files:
    print(f"\nFiles with copy errors:")
    for animal_id, filename, error in skipped_files:
        print(f"  {animal_id} - {filename}: {error}")

# Verify output directory
output_files = os.listdir(output_dir)
print(f"\nOutput directory verification:")
print(f"  Directory: {output_dir}")
print(f"  Total files in output: {len(output_files)}")
print(f"  Expected files: {len(copied_files)}")

# Check if any target animals are completely missing
print(f"\nMissing animals check:")
animals_found = set(animal_id for animal_id, _ in copied_files)
missing_animals = set(target_animal_ids) - animals_found

if missing_animals:
    print(f"Animals with NO files found ({len(missing_animals)}):")
    for animal_id in sorted(missing_animals):
        print(f"  - {animal_id}")
else:
    print("✓ All target animals have at least one file")

print(f"\n✓ Copy operation completed!")
print(f"Files copied to: {output_dir}")

已加载 34 个on-nest文件
已加载 34 个licking文件
已加载 31 个nursing文件
已加载 34 个self-groom文件

on-nest文件键示例:
  C10_ELS26 -> C10_ELS26.xlsx
  C11_ELS29 -> C11_ELS29.xlsx
  C1_ELS1 -> C1_ELS1.xlsx

licking文件键示例:
  C10_ELS26 -> C10_ELS26.xlsx
  C11_ELS29 -> C11_ELS29.xlsx
  C1_ELS1 -> C1_ELS1.xlsx

nursing文件键示例:
  C10_ELS26 -> C10_ELS26.xlsx
  C11_ELS29 -> C11_ELS29.xlsx
  C1_ELS1 -> C1_ELS1.xlsx

self-groom文件键示例:
  C10_ELS26 -> C10_ELS26.xlsx
  C11_ELS29 -> C11_ELS29.xlsx
  C1_ELS1 -> C1_ELS1.xlsx

处理 /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/MouseC10F4ELS26_P3.pkl
  提取的键: C10_ELS26
  总窗口数: 3677
  总持续时间: 11031.0 秒

  找到on-nest文件: C10_ELS26.xlsx
  on-nest行为记录: 23 条
  on-nest行为时间范围: 316.044 - 10800.0 秒
    总窗口数: 3677, 总时长: 11031.0 秒
    行为记录范围: 316.044 秒至 10800.0 秒
    行为记录数量: 23
    标记为1的窗口数量: 2103
    第一个标记为1的窗口: 105 (时间: 315.0-318.0 秒)
    最后一个标记为1的窗口: 3599 (时间: 10797.0-10800.0 秒)
  添加了onnest_raw标签，有 2103 个标记为1的窗口

  找到licking文件: C10_ELS26.xlsx
  licking行为记录: 14 条
  lick

  nursing行为记录: 7 条
  nursing行为时间范围: 1597.209 - 10800.0 秒
    总窗口数: 4383, 总时长: 13149.0 秒
    行为记录范围: 1597.209 秒至 10800.0 秒
    行为记录数量: 7
    标记为1的窗口数量: 1927
    第一个标记为1的窗口: 532 (时间: 1596.0-1599.0 秒)
    最后一个标记为1的窗口: 3599 (时间: 10797.0-10800.0 秒)
  添加了nursing标签，有 1927 个标记为1的窗口

  找到self-groom文件: C3_ELS36.xlsx
  self-groom行为记录: 26 条
  self-groom行为时间范围: 176.415 - 10612.994 秒
    总窗口数: 4383, 总时长: 13149.0 秒
    行为记录范围: 176.415 秒至 10612.994 秒
    行为记录数量: 26
    标记为1的窗口数量: 285
    第一个标记为1的窗口: 59 (时间: 177.0-180.0 秒)
    最后一个标记为1的窗口: 3537 (时间: 10611.0-10614.0 秒)
  添加了selfgroom标签，有 285 个标记为1的窗口

  已成功保存更新后的PKL文件

处理 /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/MouseC4F4ELS38_P3.pkl
  提取的键: C4_ELS38
  总窗口数: 5266
  总持续时间: 15798.0 秒

  找到on-nest文件: C4_ELS38.xlsx
  on-nest行为记录: 160 条
  on-nest行为时间范围: 74.69 - 10800.0 秒
    总窗口数: 5266, 总时长: 15798.0 秒
    行为记录范围: 74.69 秒至 10800.0 秒
    行为记录数量: 160
    标记为1的窗口数量: 1836
    第一个标记为1的窗口: 39 (时间: 117.0-120.0 秒)
    最后一个标记为1的窗口: 

  nursing行为记录: 4 条
  nursing行为时间范围: 1726.81 - 10800.0 秒
    总窗口数: 4801, 总时长: 14403.0 秒
    行为记录范围: 1726.81 秒至 10800.0 秒
    行为记录数量: 4
    标记为1的窗口数量: 2371
    第一个标记为1的窗口: 576 (时间: 1728.0-1731.0 秒)
    最后一个标记为1的窗口: 3599 (时间: 10797.0-10800.0 秒)
  添加了nursing标签，有 2371 个标记为1的窗口

  找到self-groom文件: C7_ELS22.xlsx
  self-groom行为记录: 25 条
  self-groom行为时间范围: 0.0 - 4149.466 秒
    总窗口数: 4801, 总时长: 14403.0 秒
    行为记录范围: 0.0 秒至 4149.466 秒
    行为记录数量: 25
    标记为1的窗口数量: 89
    第一个标记为1的窗口: 0 (时间: 0.0-3.0 秒)
    最后一个标记为1的窗口: 1201 (时间: 3603.0-3606.0 秒)
  添加了selfgroom标签，有 89 个标记为1的窗口

  已成功保存更新后的PKL文件

处理 /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/MouseC7F2ELS45_P3.pkl
  提取的键: C7_ELS45
  总窗口数: 3695
  总持续时间: 11085.0 秒

  找到on-nest文件: C7_ELS45.xlsx
  on-nest行为记录: 30 条
  on-nest行为时间范围: 239.212 - 10772.075 秒
    总窗口数: 3695, 总时长: 11085.0 秒
    行为记录范围: 239.212 秒至 10772.075 秒
    行为记录数量: 30
    标记为1的窗口数量: 872
    第一个标记为1的窗口: 80 (时间: 240.0-243.0 秒)
    最后一个标记为1的窗口: 3590 (时间: 10770.

  self-groom行为记录: 54 条
  self-groom行为时间范围: 266.293 - 10583.332 秒
    总窗口数: 4123, 总时长: 12369.0 秒
    行为记录范围: 266.293 秒至 10583.332 秒
    行为记录数量: 54
    标记为1的窗口数量: 176
    第一个标记为1的窗口: 89 (时间: 267.0-270.0 秒)
    最后一个标记为1的窗口: 3527 (时间: 10581.0-10584.0 秒)
  添加了selfgroom标签，有 176 个标记为1的窗口

  已成功保存更新后的PKL文件

处理 /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/MouseE1F4ELS33_P3.pkl
  提取的键: E1_ELS33
  总窗口数: 3615
  总持续时间: 10845.0 秒

  找到on-nest文件: E1_ELS33.xlsx
  on-nest行为记录: 70 条
  on-nest行为时间范围: 203.511 - 10800.0 秒
    总窗口数: 3615, 总时长: 10845.0 秒
    行为记录范围: 203.511 秒至 10800.0 秒
    行为记录数量: 70
    标记为1的窗口数量: 2230
    第一个标记为1的窗口: 68 (时间: 204.0-207.0 秒)
    最后一个标记为1的窗口: 3599 (时间: 10797.0-10800.0 秒)
  添加了onnest_raw标签，有 2230 个标记为1的窗口

  找到licking文件: E1_ELS33.xlsx
  licking行为记录: 51 条
  licking行为时间范围: 221.924 - 10624.98 秒
    总窗口数: 3615, 总时长: 10845.0 秒
    行为记录范围: 221.924 秒至 10624.98 秒
    行为记录数量: 51
    标记为1的窗口数量: 155
    第一个标记为1的窗口: 74 (时间: 222.0-225.0 秒)
    最后一个标记为1的窗口: 


  已成功保存更新后的PKL文件

处理 /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/MouseE5ELS8_P3.pkl
  提取的键: E5_ELS8
  总窗口数: 3673
  总持续时间: 11019.0 秒

  找到on-nest文件: E5_ELS8.xlsx
  on-nest行为记录: 36 条
  on-nest行为时间范围: 81.402 - 10705.266 秒
    总窗口数: 3673, 总时长: 11019.0 秒
    行为记录范围: 81.402 秒至 10705.266 秒
    行为记录数量: 36
    标记为1的窗口数量: 749
    第一个标记为1的窗口: 27 (时间: 81.0-84.0 秒)
    最后一个标记为1的窗口: 3567 (时间: 10701.0-10704.0 秒)
  添加了onnest_raw标签，有 749 个标记为1的窗口

  找到licking文件: E5_ELS8.xlsx
  licking行为记录: 8 条
  licking行为时间范围: 135.253 - 10303.253 秒
    总窗口数: 3673, 总时长: 11019.0 秒
    行为记录范围: 135.253 秒至 10303.253 秒
    行为记录数量: 8
    标记为1的窗口数量: 26
    第一个标记为1的窗口: 45 (时间: 135.0-138.0 秒)
    最后一个标记为1的窗口: 3433 (时间: 10299.0-10302.0 秒)
  添加了licking标签，有 26 个标记为1的窗口

  找到nursing文件: E5_ELS8.xlsx
  nursing行为记录: 1 条
  nursing行为时间范围: 3410.919 - 4917.7 秒
    总窗口数: 3673, 总时长: 11019.0 秒
    行为记录范围: 3410.919 秒至 4917.7 秒
    行为记录数量: 1
    标记为1的窗口数量: 502
    第一个标记为1的窗口: 1137 (时间: 3411.0-3414.0 秒)
    最后一个


  已成功保存更新后的PKL文件

总共成功更新了 30 个文件

成功更新的文件列表:
1. MouseC10F4ELS26_P3.pkl
2. MouseC11F1ELS29_P3.pkl
3. MouseC1ELS16_P3.pkl
4. MouseC1F3ELS32_P3.pkl
5. MouseC2F4ELS18_P3.pkl
6. MouseC3F3ELS36_P3.pkl
7. MouseC4F4ELS38_P3.pkl
8. MouseC5F3ELS40_P3.pkl
9. MouseC5F4ELS20_P3.pkl
10. MouseC6F5ELS42_P3.pkl
11. MouseC7ELS11_P3.pkl
12. MouseC7F1ELS22_P3.pkl
13. MouseC7F2ELS45_P3.pkl
14. MouseC7F4ELS12_P3.pkl
15. MouseC8ELS13_P3.pkl
16. MouseC9F3ELS24_P3.pkl
17. MouseE11F2ELS28_P3.pkl
18. MouseE12F4ELS31_P3.pkl
19. MouseE1F4ELS33_P3.pkl
20. MouseE2ELS3_P3.pkl
21. MouseE3ELS7_P3.pkl
22. MouseE3F1ELS37_P3.pkl
23. MouseE4F2ELS39_P3.pkl
24. MouseE4F3ELS19_P3.pkl
25. MouseE5ELS8_P3.pkl
26. MouseE5F1ELS41_P3.pkl
27. MouseE6F4ELS44_P3.pkl
28. MouseE7F2ELS21_P3.pkl
29. MouseE8F4ELS23_P3.pkl
30. MouseE9ELS15_P3.pkl


### Aggregate per-mouse `_P3.pkl` with licking labels → `full_onnest_lick.pkl` (recovered from older preproc)

In [ ]:
all_onnest_data = {
    'power': [],
    'coh_sq_coherence': [],
    'freq_band': None,
    'region': None,
    'region_pair': None,
    'mouse_id': [],
    'period': [],
    'onnest_raw': [],
    'licking':[]
}

# Load and combine data from each file
print("Loading and combining onnest feature files...")
for i, file_path in enumerate(p3_updated_fns):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
        
        # Append array data
        all_onnest_data['power'].append(data['power'])
        all_onnest_data['coh_sq_coherence'].append(data['coh_sq_coherence'])
        all_onnest_data['mouse_id'].append(data['mouse_id'])
        all_onnest_data['period'].append(data['period'])
        all_onnest_data['onnest_raw'].append(data['onnest_raw'])
        all_onnest_data['licking'].append(data['licking'])
        
        # Store static data from first file
        if i == 0:
            all_onnest_data['freq_band'] = data['freq_band']
            all_onnest_data['region'] = data['region']
            all_onnest_data['region_pair'] = data['region_pair']
            
    if (i+1) % 10 == 0:
        print(f"  Processed {i+1}/{len((p3_updated_fns))} files")

# Concatenate all the array data
print("\nConcatenating arrays...")
all_onnest_data['power'] = np.concatenate(all_onnest_data['power'], axis=0)
all_onnest_data['coh_sq_coherence'] = np.concatenate(all_onnest_data['coh_sq_coherence'], axis=0)
all_onnest_data['mouse_id'] = np.concatenate(all_onnest_data['mouse_id'], axis=0)
all_onnest_data['period'] = np.concatenate(all_onnest_data['period'], axis=0)
all_onnest_data['onnest_raw'] = np.concatenate(all_onnest_data['onnest_raw'], axis=0)
all_onnest_data['licking'] = np.concatenate(all_onnest_data['licking'], axis=0)

# Save the combined data
output_path = os.path.join("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick", "full_onnest_lick.pkl")
print(f"Saving combined onnest data to {output_path}...")
with open(output_path, 'wb') as file:
    pickle.dump(all_onnest_data, file)

# Print summary
print("\n" + "-" * 80)
print(f"ONNEST COMBINATION COMPLETE")
print(f"  Total files processed: {len((p3_updated_fns))}")
print(f"  Output file: {output_path}")
print(f"  Data shape summary:")
print(f"    - power: {all_onnest_data['power'].shape}")
print(f"    - coh_sq_coherence: {all_onnest_data['coh_sq_coherence'].shape}")
print(f"    - mouse_id: {all_onnest_data['mouse_id'].shape}")
print(f"    - period: {all_onnest_data['period'].shape}")
print(f"    - onnest_raw: {all_onnest_data['onnest_raw'].shape}")
print(f"    - licking: {all_onnest_data['licking'].shape}")
print("-" * 80)

In [20]:
all_data_with_nursing = {
    'power': [],
    'coh_sq_coherence': [],
    'freq_band': None,
    'region': None,
    'region_pair': None,
    'mouse_id': [],
    'period': [],
    'onnest_raw': [],
    'licking': [],
    'nursing': [],
    'selfgroom': []
}

all_data_without_nursing = {
    'power': [],
    'coh_sq_coherence': [],
    'freq_band': None,
    'region': None,
    'region_pair': None,
    'mouse_id': [],
    'period': [],
    'onnest_raw': [],
    'licking': [],
    'selfgroom': []
}

# Track files with missing nursing data
files_without_nursing = []
files_with_nursing = 0
files_without_nursing_count = 0

# Load and combine data from each file
print("Loading and combining behavior feature files...")
for i, file_path in enumerate(p3_updated_fns):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
        
        # Check if nursing exists
        has_nursing = 'nursing' in data
        
        if has_nursing:
            # Add to dataset WITH nursing
            all_data_with_nursing['power'].append(data['power'])
            all_data_with_nursing['coh_sq_coherence'].append(data['coh_sq_coherence'])
            all_data_with_nursing['mouse_id'].append(data['mouse_id'])
            all_data_with_nursing['period'].append(data['period'])
            all_data_with_nursing['onnest_raw'].append(data['onnest_raw'])
            all_data_with_nursing['licking'].append(data['licking'])
            all_data_with_nursing['nursing'].append(data['nursing'])
            all_data_with_nursing['selfgroom'].append(data['selfgroom'])
            
            # Store static data from first file with nursing
            if files_with_nursing == 0:
                all_data_with_nursing['freq_band'] = data['freq_band']
                all_data_with_nursing['region'] = data['region']
                all_data_with_nursing['region_pair'] = data['region_pair']
            
            files_with_nursing += 1
        else:
            files_without_nursing.append(os.path.basename(file_path))
            print(f"  File without nursing: {os.path.basename(file_path)}")
        
        # Add to dataset WITHOUT nursing (all files)
        all_data_without_nursing['power'].append(data['power'])
        all_data_without_nursing['coh_sq_coherence'].append(data['coh_sq_coherence'])
        all_data_without_nursing['mouse_id'].append(data['mouse_id'])
        all_data_without_nursing['period'].append(data['period'])
        all_data_without_nursing['onnest_raw'].append(data['onnest_raw'])
        all_data_without_nursing['licking'].append(data['licking'])
        all_data_without_nursing['selfgroom'].append(data['selfgroom'])
        
        # Store static data from first file
        if i == 0:
            all_data_without_nursing['freq_band'] = data['freq_band']
            all_data_without_nursing['region'] = data['region']
            all_data_without_nursing['region_pair'] = data['region_pair']
            
    if (i+1) % 10 == 0:
        print(f"  Processed {i+1}/{len(p3_updated_fns)} files")

# Concatenate arrays for dataset WITH nursing
print("\nConcatenating arrays for dataset with nursing...")
all_data_with_nursing['power'] = np.concatenate(all_data_with_nursing['power'], axis=0)
all_data_with_nursing['coh_sq_coherence'] = np.concatenate(all_data_with_nursing['coh_sq_coherence'], axis=0)
all_data_with_nursing['mouse_id'] = np.concatenate(all_data_with_nursing['mouse_id'], axis=0)
all_data_with_nursing['period'] = np.concatenate(all_data_with_nursing['period'], axis=0)
all_data_with_nursing['onnest_raw'] = np.concatenate(all_data_with_nursing['onnest_raw'], axis=0)
all_data_with_nursing['licking'] = np.concatenate(all_data_with_nursing['licking'], axis=0)
all_data_with_nursing['nursing'] = np.concatenate(all_data_with_nursing['nursing'], axis=0)
all_data_with_nursing['selfgroom'] = np.concatenate(all_data_with_nursing['selfgroom'], axis=0)

# Concatenate arrays for dataset WITHOUT nursing
print("Concatenating arrays for dataset without nursing field...")
all_data_without_nursing['power'] = np.concatenate(all_data_without_nursing['power'], axis=0)
all_data_without_nursing['coh_sq_coherence'] = np.concatenate(all_data_without_nursing['coh_sq_coherence'], axis=0)
all_data_without_nursing['mouse_id'] = np.concatenate(all_data_without_nursing['mouse_id'], axis=0)
all_data_without_nursing['period'] = np.concatenate(all_data_without_nursing['period'], axis=0)
all_data_without_nursing['onnest_raw'] = np.concatenate(all_data_without_nursing['onnest_raw'], axis=0)
all_data_without_nursing['licking'] = np.concatenate(all_data_without_nursing['licking'], axis=0)
all_data_without_nursing['selfgroom'] = np.concatenate(all_data_without_nursing['selfgroom'], axis=0)

# Create output directory if it doesn't exist
output_dir = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse"
os.makedirs(output_dir, exist_ok=True)

# Save dataset WITH nursing (only files that have nursing)
output_path_with_nursing = os.path.join(output_dir, "full_all_behaviors_with_nursing.pkl")
print(f"\nSaving dataset WITH nursing to {output_path_with_nursing}...")
with open(output_path_with_nursing, 'wb') as file:
    pickle.dump(all_data_with_nursing, file)

# Save dataset WITHOUT nursing field (all files)
output_path_without_nursing = os.path.join(output_dir, "full_all_behaviors_no_nursing_field.pkl")
print(f"Saving dataset WITHOUT nursing field to {output_path_without_nursing}...")
with open(output_path_without_nursing, 'wb') as file:
    pickle.dump(all_data_without_nursing, file)

# Print summary
print("\n" + "=" * 80)
print(f"BEHAVIOR DATA COMBINATION COMPLETE")
print(f"\n  Total files processed: {len(p3_updated_fns)}")
print(f"  Files with nursing data: {files_with_nursing}")
print(f"  Files without nursing data: {len(files_without_nursing)}")

print(f"\n" + "-" * 80)
print(f"DATASET 1: WITH NURSING (n={files_with_nursing} files)")
print(f"  File: full_all_behaviors_with_nursing.pkl")
print(f"  Data shape summary:")
print(f"    - power: {all_data_with_nursing['power'].shape}")
print(f"    - coh_sq_coherence: {all_data_with_nursing['coh_sq_coherence'].shape}")
print(f"    - mouse_id: {all_data_with_nursing['mouse_id'].shape}")
print(f"    - period: {all_data_with_nursing['period'].shape}")
print(f"    - onnest_raw: {all_data_with_nursing['onnest_raw'].shape}")
print(f"    - licking: {all_data_with_nursing['licking'].shape}")
print(f"    - nursing: {all_data_with_nursing['nursing'].shape}")
print(f"    - selfgroom: {all_data_with_nursing['selfgroom'].shape}")

print(f"\n" + "-" * 80)
print(f"DATASET 2: WITHOUT NURSING FIELD (n={len(p3_updated_fns)} files)")
print(f"  File: full_all_behaviors_no_nursing_field.pkl")
print(f"  Data shape summary:")
print(f"    - power: {all_data_without_nursing['power'].shape}")
print(f"    - coh_sq_coherence: {all_data_without_nursing['coh_sq_coherence'].shape}")
print(f"    - mouse_id: {all_data_without_nursing['mouse_id'].shape}")
print(f"    - period: {all_data_without_nursing['period'].shape}")
print(f"    - onnest_raw: {all_data_without_nursing['onnest_raw'].shape}")
print(f"    - licking: {all_data_without_nursing['licking'].shape}")
print(f"    - selfgroom: {all_data_without_nursing['selfgroom'].shape}")

# Print the files without nursing data
if files_without_nursing:
    print(f"\n" + "-" * 80)
    print(f"FILES WITHOUT NURSING DATA ({len(files_without_nursing)}):")
    for idx, filename in enumerate(files_without_nursing, 1):
        print(f"  {idx}. {filename}")
print("=" * 80)

Loading and combining behavior feature files...
  Processed 10/30 files
  File without nursing: MouseE12F4ELS31_P3.pkl
  File without nursing: MouseE2ELS3_P3.pkl
  Processed 20/30 files
  File without nursing: MouseE5F1ELS41_P3.pkl
  Processed 30/30 files

Concatenating arrays for dataset with nursing...
Concatenating arrays for dataset without nursing field...

Saving dataset WITH nursing to /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_with_nursing.pkl...
Saving dataset WITHOUT nursing field to /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field.pkl...

BEHAVIOR DATA COMBINATION COMPLETE

  Total files processed: 30
  Files with nursing data: 27
  Files without nursing data: 3

--------------------------------------------------------------------------------
DATASET 1: WITH NURSING (n=27 files)
  

# 8. Pups Retrieval -- Oct22

+ P3 and P4 certain animals has pup retrieval data (not all -- subset): "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3P4_pup_retrieval.pkl"

In [4]:
P4_pup_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P4 Recordings/Home cage pup retrieval/Scoring of Pup Retrieval Behavior",
                           ".xlsx")

P3_pup_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/pup-retrieval",
                              "xlsx")

11 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P4 Recordings/Home cage pup retrieval/Scoring of Pup Retrieval Behavior
14 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/P3 Recordings/Behavioral Scoring/maternal behaviors/pup-retrieval


In [5]:
all_fns = get_filenames('/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All', 'pkl')
p3_fns = [fn for fn in all_fns if fn.endswith('_P3.pkl')]
p4_fns = [fn for fn in all_fns if fn.endswith('_P4 home.pkl')]

246 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All


### modify the pup retrieval - Oct30

In [20]:
p4_home_fns = [fn for fn in all_fns if fn.endswith('_P4 home.pkl')]

In [22]:
import os
import shutil
import re
from pathlib import Path

# Target Animal IDs from Filter 1 (all 8 regions YES)
target_animal_ids = [
    'C1_ELS1', 'C6_ELS9', 'C7_ELS11', 'E2_ELS3', 'C2_ELS18', 'C5_ELS20', 
    'C7_ELS22', 'C1_ELS32', 'C5_ELS40', 'C6_ELS42', 'C7_ELS45', 'E1_ELS33', 
    'E2_ELS35', 'E3_ELS37', 'E4_ELS39', 'E5_ELS41', 'E6_ELS43', 'E6_ELS44'
]

# Output directory
output_dir = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes"
os.makedirs(output_dir, exist_ok=True)

def extract_animal_id_from_filename(file_path):
    """
    Extract animal ID from file path, ignoring FXX part
    Examples:
    - MouseC1F3ELS32_Ges.pkl -> C1_ELS32
    - MouseE6F4ELS44_P1.pkl -> E6_ELS44
    - MouseC7ELS11_P3.pkl -> C7_ELS11
    """
    filename = Path(file_path).stem
    filename = filename.replace('Mouse', '')
    
    # Split by underscore to get animal part
    animal_part = filename.split('_')[0]
    
    # Extract animal ID using regex (ignore F part)
    match = re.match(r'([CE]\d+)(?:F\d+)?(ELS\d+)', animal_part)
    if match:
        prefix, els_part = match.groups()
        return f"{prefix}_{els_part}"
    return None

def should_include_file(file_path):
    """
    Check if file should be included (exclude Pre pup and P4 open)
    """
    filename = Path(file_path).stem
    
    # Exclude Pre pup and P4 open files
    if 'Pre pup' in filename or 'P4 open' in filename:
        return False
    return True

# Find and copy files for target animals
copied_files = []
skipped_files = []
excluded_files = []

print(f"Target animals: {len(target_animal_ids)}")
print(f"Total files to check: {len(all_fns)}")
print(f"Output directory: {output_dir}")
print("Excluding: Pre pup and P4 open files")
print("\nProcessing files...")

for file_path in all_fns:
    # Extract animal ID from filename
    animal_id = extract_animal_id_from_filename(file_path)
    
    if animal_id in target_animal_ids:
        # Check if file should be included (exclude Pre pup and P4 open)
        if should_include_file(file_path):
            # Copy file to output directory
            source_path = file_path
            filename = Path(file_path).name
            dest_path = os.path.join(output_dir, filename)
            
            try:
                shutil.copy2(source_path, dest_path)
                copied_files.append((animal_id, filename))
                print(f"✓ Copied: {filename} (Animal: {animal_id})")
            except Exception as e:
                print(f"✗ Error copying {filename}: {e}")
                skipped_files.append((animal_id, filename, str(e)))
        else:
            # File excluded due to stage filter
            filename = Path(file_path).name
            excluded_files.append((animal_id, filename))
            print(f"- Excluded: {filename} (Animal: {animal_id}) - Pre pup/P4 open")
    else:
        # File not in target list - skip
        continue

# Summary report
print(f"\n" + "="*80)
print("COPY OPERATION SUMMARY")
print("="*80)
print(f"Target animals: {len(target_animal_ids)}")
print(f"Files successfully copied: {len(copied_files)}")
print(f"Files excluded (Pre pup/P4 open): {len(excluded_files)}")
print(f"Files with errors: {len(skipped_files)}")

# Group by animal ID to show completeness
print(f"\nFiles copied per animal:")
from collections import defaultdict
files_per_animal = defaultdict(list)

for animal_id, filename in copied_files:
    files_per_animal[animal_id].append(filename)

for animal_id in sorted(target_animal_ids):
    if animal_id in files_per_animal:
        file_count = len(files_per_animal[animal_id])
        print(f"  {animal_id:<12}: {file_count:2d} files")
        # Show first few filenames as examples
        for i, filename in enumerate(sorted(files_per_animal[animal_id])[:3]):
            print(f"    - {filename}")
        if file_count > 3:
            print(f"    ... and {file_count - 3} more files")
    else:
        print(f"  {animal_id:<12}: 0 files (NOT FOUND)")

# Show excluded files summary
if excluded_files:
    print(f"\nExcluded files (Pre pup/P4 open):")
    excluded_per_animal = defaultdict(list)
    for animal_id, filename in excluded_files:
        excluded_per_animal[animal_id].append(filename)
    
    for animal_id in sorted(excluded_per_animal.keys()):
        file_count = len(excluded_per_animal[animal_id])
        print(f"  {animal_id}: {file_count} files excluded")

# Show any errors
if skipped_files:
    print(f"\nFiles with copy errors:")
    for animal_id, filename, error in skipped_files:
        print(f"  {animal_id} - {filename}: {error}")

# Verify output directory
output_files = os.listdir(output_dir)
print(f"\nOutput directory verification:")
print(f"  Directory: {output_dir}")
print(f"  Total files in output: {len(output_files)}")
print(f"  Expected files: {len(copied_files)}")

# Check if any target animals are completely missing
print(f"\nMissing animals check:")
animals_found = set(animal_id for animal_id, _ in copied_files)
missing_animals = set(target_animal_ids) - animals_found

if missing_animals:
    print(f"Animals with NO files found ({len(missing_animals)}):")
    for animal_id in sorted(missing_animals):
        print(f"  - {animal_id}")
else:
    print("✓ All target animals have at least one file")

print(f"\n✓ Copy operation completed!")
print(f"Files copied to: {output_dir}")

开始处理P4 Home数据...

处理 P4 Home 数据
已加载 9 个pup_retrieval文件
已加载 11 个trial时间表
总共有 10 个PKL文件

pup_retrieval文件键示例:
  C1_ELS32 -> C1_ELS32.xlsx
  C3_ELS36 -> C3_ELS36.xlsx
  C5_ELS40 -> C5_ELS40.xlsx
  C6_ELS42 -> C6_ELS42.xlsx
  C7_ELS45 -> C7_ELS45.xlsx

trial时间表键示例:
  C1_ELS32 -> 10 trials
  C3_ELS36 -> 10 trials
  C4_ELS38 -> 10 trials
  C5_ELS40 -> 10 trials
  C6_ELS42 -> 10 trials

PKL文件名示例:
  MouseC1F3ELS32_P4 home.pkl -> 键: C1_ELS32
  MouseC3F3ELS36_P4 home.pkl -> 键: C3_ELS36
  MouseC5F3ELS40_P4 home.pkl -> 键: C5_ELS40
  MouseC6F5ELS42_P4 home.pkl -> 键: C6_ELS42
  MouseC7F2ELS45_P4 home.pkl -> 键: C7_ELS45

处理 MouseC1F3ELS32_P4 home.pkl
  提取的键: C1_ELS32
  总窗口数: 759
  总持续时间: 2277.0 秒
  找到pup_retrieval文件: C1_ELS32.xlsx
  Retrieval行为记录: 6 条
    总窗口数: 759, 总时长: 2277.0 秒
    Trial记录数量: 10
    Trial时间范围: 1261.77 - 2262.95 秒
    Retrieval行为记录数量: 6
    Partial retrieval: 0, Successful retrieval: 6
    标签分布:
      0 (no trial): 474 窗口
      1 (trial no retrieval): 272 窗口
      3 (partial retrieva

In [23]:
import os
import shutil
import re
from pathlib import Path

# Target Animal IDs from Filter 1 (all 8 regions YES)
target_animal_ids = [
    'C1_ELS1', 'C6_ELS9', 'C7_ELS11', 'E2_ELS3', 'C2_ELS18', 'C5_ELS20', 
    'C7_ELS22', 'C1_ELS32', 'C5_ELS40', 'C6_ELS42', 'C7_ELS45', 'E1_ELS33', 
    'E2_ELS35', 'E3_ELS37', 'E4_ELS39', 'E5_ELS41', 'E6_ELS43', 'E6_ELS44'
]

# Output directory
output_dir = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes"
os.makedirs(output_dir, exist_ok=True)

def extract_animal_id_from_filename(file_path):
    """
    Extract animal ID from file path, ignoring FXX part
    Examples:
    - MouseC1F3ELS32_Ges.pkl -> C1_ELS32
    - MouseE6F4ELS44_P1.pkl -> E6_ELS44
    - MouseC7ELS11_P3.pkl -> C7_ELS11
    """
    filename = Path(file_path).stem
    filename = filename.replace('Mouse', '')
    
    # Split by underscore to get animal part
    animal_part = filename.split('_')[0]
    
    # Extract animal ID using regex (ignore F part)
    match = re.match(r'([CE]\d+)(?:F\d+)?(ELS\d+)', animal_part)
    if match:
        prefix, els_part = match.groups()
        return f"{prefix}_{els_part}"
    return None

def should_include_file(file_path):
    """
    Check if file should be included (exclude Pre pup and P4 open)
    """
    filename = Path(file_path).stem
    
    # Exclude Pre pup and P4 open files
    if 'Pre pup' in filename or 'P4 open' in filename:
        return False
    return True

# Find and copy files for target animals
copied_files = []
skipped_files = []
excluded_files = []

print(f"Target animals: {len(target_animal_ids)}")
print(f"Total files to check: {len(all_fns)}")
print(f"Output directory: {output_dir}")
print("Excluding: Pre pup and P4 open files")
print("\nProcessing files...")

for file_path in all_fns:
    # Extract animal ID from filename
    animal_id = extract_animal_id_from_filename(file_path)
    
    if animal_id in target_animal_ids:
        # Check if file should be included (exclude Pre pup and P4 open)
        if should_include_file(file_path):
            # Copy file to output directory
            source_path = file_path
            filename = Path(file_path).name
            dest_path = os.path.join(output_dir, filename)
            
            try:
                shutil.copy2(source_path, dest_path)
                copied_files.append((animal_id, filename))
                print(f"✓ Copied: {filename} (Animal: {animal_id})")
            except Exception as e:
                print(f"✗ Error copying {filename}: {e}")
                skipped_files.append((animal_id, filename, str(e)))
        else:
            # File excluded due to stage filter
            filename = Path(file_path).name
            excluded_files.append((animal_id, filename))
            print(f"- Excluded: {filename} (Animal: {animal_id}) - Pre pup/P4 open")
    else:
        # File not in target list - skip
        continue

# Summary report
print(f"\n" + "="*80)
print("COPY OPERATION SUMMARY")
print("="*80)
print(f"Target animals: {len(target_animal_ids)}")
print(f"Files successfully copied: {len(copied_files)}")
print(f"Files excluded (Pre pup/P4 open): {len(excluded_files)}")
print(f"Files with errors: {len(skipped_files)}")

# Group by animal ID to show completeness
print(f"\nFiles copied per animal:")
from collections import defaultdict
files_per_animal = defaultdict(list)

for animal_id, filename in copied_files:
    files_per_animal[animal_id].append(filename)

for animal_id in sorted(target_animal_ids):
    if animal_id in files_per_animal:
        file_count = len(files_per_animal[animal_id])
        print(f"  {animal_id:<12}: {file_count:2d} files")
        # Show first few filenames as examples
        for i, filename in enumerate(sorted(files_per_animal[animal_id])[:3]):
            print(f"    - {filename}")
        if file_count > 3:
            print(f"    ... and {file_count - 3} more files")
    else:
        print(f"  {animal_id:<12}: 0 files (NOT FOUND)")

# Show excluded files summary
if excluded_files:
    print(f"\nExcluded files (Pre pup/P4 open):")
    excluded_per_animal = defaultdict(list)
    for animal_id, filename in excluded_files:
        excluded_per_animal[animal_id].append(filename)
    
    for animal_id in sorted(excluded_per_animal.keys()):
        file_count = len(excluded_per_animal[animal_id])
        print(f"  {animal_id}: {file_count} files excluded")

# Show any errors
if skipped_files:
    print(f"\nFiles with copy errors:")
    for animal_id, filename, error in skipped_files:
        print(f"  {animal_id} - {filename}: {error}")

# Verify output directory
output_files = os.listdir(output_dir)
print(f"\nOutput directory verification:")
print(f"  Directory: {output_dir}")
print(f"  Total files in output: {len(output_files)}")
print(f"  Expected files: {len(copied_files)}")

# Check if any target animals are completely missing
print(f"\nMissing animals check:")
animals_found = set(animal_id for animal_id, _ in copied_files)
missing_animals = set(target_animal_ids) - animals_found

if missing_animals:
    print(f"Animals with NO files found ({len(missing_animals)}):")
    for animal_id in sorted(missing_animals):
        print(f"  - {animal_id}")
else:
    print("✓ All target animals have at least one file")

print(f"\n✓ Copy operation completed!")
print(f"Files copied to: {output_dir}")

开始合并P4数据集
  P4文件数: 10
  总文件数: 10

加载并合并pup_retrieval和pup_retrieval_detail数据...
  警告: MouseE3F1ELS37_P4 home.pkl 没有pup_retrieval字段
  已处理 10/10 个文件

拼接数组...

保存数据集到 /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P4_pup_retrieval_detail.pkl...

P4数据集合并完成

输出文件: P4_pup_retrieval_detail.pkl
保存路径: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P4_pup_retrieval_detail.pkl

处理统计:
  成功包含的文件数: 10
  缺少pup_retrieval的文件数: 1
  缺少pup_retrieval_detail的文件数: 0

数据形状:
  power: (8263, 24)
  coh_sq_coherence: (8263, 84)
  mouse_id: (8263,)
  period: (8263,)
  pup_retrieval: (8263,)
  pup_retrieval_detail: (8263,)

pup_retrieval标签分布:
  标签 0 (no retrieval): 8112 个窗口 (0.9817)
  标签 1 (retrieval): 151 个窗口 (0.0183)

pup_retrieval_detail标签分布:
  标签 0 (no trial): 6211 个窗口 (0.7517)
  标签 1 (trial no retrieval): 1901 个窗口 (0.2301)
  标签 3 (partial retrieval): 19 个窗口 (0.0023)
  标签 4 (successful retrieval): 132 个窗口 (0.0160)

缺少pup_retrieval的文件列表

# 9. Check dataset consistency and Trim LFP 

+ 1. "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi.pkl"
+ 2. "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi.pkl"
+ 3. "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick/full_onnest_lick.pkl"
+ 4. "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field.pkl"
+ 5. "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P4_pup_retrieval_detail.pkl"

4/3/2/1-hour timepoints for the P1/3/8/14 data; first 20 mins for P4 home; 10 mins for Pre home /Ges/P20

+ 1 to "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi_Trim_Post.pkl", "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi_Trim_All.pkl"
+ 2 to "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Trim.pkl"
+ 3 to "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick/full_onnest_lick_Trim.pkl"
+ 4 to "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field_Trim.pkl"
+ 5 not do trim b/c we need later data for pup-retrevial experiments

### 1. Stage EF training, full stage data that other EFs use for backprojection

first projection all P4 and then P4 take only first 20 mins in plot and test function code

In [33]:
TRAINING_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi.pkl"
X_FEATURE_LIST = ["power", "coh_sq_coherence"]
X_FEATURE_WEIGHTS = [1, 1]
Y_FEATURE = "period"

with open(TRAINING_DATA_FILE, "rb") as f:
    train_dict = pickle.load(f)

unique_periods = np.unique(train_dict[Y_FEATURE])
print("unique periods:")
print(unique_periods)
print("unique mouse id:")
print(np.unique(train_dict['mouse_id']))


full_dict = train_dict
print("Dictionary keys:")
for key in full_dict.keys():
    print(key)

# Print some sample elements
print("\nSample elements:")
for key in full_dict.keys():
    value = full_dict[key]
    print(f"\nKey: {key}")
    
    # Handle different types of data
    if isinstance(value, list) and len(value) > 0:
        print(f"Type: list, Length: {len(value)}")
        print(f"First 3 elements: {value[:3]}")
    elif isinstance(value, dict):
        print(f"Type: dict, Number of keys: {len(value)}")
        print(f"Dictionary keys: {list(value.keys())[:5]}")
    elif hasattr(value, 'shape'):  # For numpy arrays
        print(f"Type: {type(value)}")
        print(f"Shape: {value.shape}")
        print(f"Data type: {value.dtype}")
        # Print a small slice of the array
        if value.size > 0:
            print("Sample (first few elements):")
            if value.ndim == 1:
                print(value[:5])  # First 5 elements for 1D array
            else:
                print(value[:2, :2])  # First 2×2 elements for 2D+ array
    elif isinstance(value, (int, float, str, bool)):
        print(f"Value: {value}")
    else:
        print(f"Type: {type(value)}")

unique periods:
['Ges' 'P1' 'P14' 'P20' 'P3' 'P4 home' 'P8' 'Pre home']
unique mouse id:
['MouseC1ELS1' 'MouseC1F3ELS32' 'MouseC2F4ELS18' 'MouseC5F3ELS40'
 'MouseC5F4ELS20' 'MouseC6ELS9' 'MouseC6F5ELS42' 'MouseC7ELS11'
 'MouseC7F1ELS22' 'MouseC7F2ELS45' 'MouseE1F4ELS33' 'MouseE2ELS3'
 'MouseE2F2ELS35' 'MouseE3F1ELS37' 'MouseE4F2ELS39' 'MouseE5F1ELS41'
 'MouseE6F1ELS43' 'MouseE6F4ELS44']
Dictionary keys:
power
coh_sq_coherence
freq_band
region
region_pair
mouse_id
period

Sample elements:

Key: power
Type: <class 'numpy.ndarray'>
Shape: (190145, 24)
Data type: float64
Sample (first few elements):
[[0.54540371 0.28907043]
 [0.41914758 0.23166547]]

Key: coh_sq_coherence
Type: <class 'numpy.ndarray'>
Shape: (190145, 84)
Data type: float64
Sample (first few elements):
[[0.87304914 0.99501394]
 [0.98118718 0.985098  ]]

Key: freq_band
Type: list, Length: 3
First 3 elements: [(2, 7), (8, 12), (14, 23)]

Key: region
Type: list, Length: 24
First 3 elements: ['BLA', 'CeA', 'IL']

Key: region_pa

In [35]:
# Define the mice IDs to keep (with F notation)
c_mice_ids = ['C6ELS9', 'C7ELS11', 'C2F4ELS18', 'C5F4ELS20', 'C7F1ELS22', 
              'C1F3ELS32', 'C5F3ELS40', 'C6F5ELS42', 'C7F2ELS45']
e_mice_ids = ['E2ELS3', 'E1F4ELS33', 'E3F1ELS37', 'E4F2ELS39', 'E5F1ELS41', 'E6F4ELS44']

# Combine the mice IDs
mice_to_keep = c_mice_ids + e_mice_ids

# Create DataFrame
df = pd.DataFrame({
    'mouse_id': full_dict['mouse_id'],
    'period': full_dict['period']
})

# Extract the short mouse ID format (remove 'Mouse' prefix)
df['mouse_id_short'] = df['mouse_id'].str.replace('Mouse', '')

# Filter to keep only specified mice
df_filtered = df[df['mouse_id_short'].isin(mice_to_keep)]

# Define the desired order for periods/stages
stage_order = ['Pre home', 'Ges', 'P1', 'P3', 'P4 home', 'P8', 'P14', 'P20']

# Create crosstab
crosstab = pd.crosstab(df_filtered['mouse_id_short'], df_filtered['period'])

# Reorder columns according to stage_order (only include stages that exist in data)
existing_stages = [stage for stage in stage_order if stage in crosstab.columns]
crosstab = crosstab[existing_stages]

# Reorder rows to match the order in c_mice_ids and e_mice_ids
existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab.index]
crosstab = crosstab.loc[existing_mice]

print("Sample counts for each mouse_id and stage combination:")
print("=" * 100)
print(crosstab)

Sample counts for each mouse_id and stage combination:
period          Pre home  Ges    P1    P3  P4 home    P8   P14   P20
mouse_id_short                                                      
C6ELS9               203  212     0     0        0     0     0   213
C7ELS11              204  259     0  3604        0  3356  1304   212
C2F4ELS18            208  216  4908  3642        0  3178  1278   205
C5F4ELS20            213  234  5009  3692        0  2524  1311   207
C7F1ELS22            210  235  4862  4801        0  2489  1354   221
C1F3ELS32            404  411  4826  3756      759  2821  1258  1561
C5F3ELS40            402  445  5110  3625      600  2763  1562   486
C6F5ELS42            410  443  5289  4308      765  2610  1280   409
C7F2ELS45            403  482  4838  3695      819  2443  1797   414
E2ELS3               267  223     0  3785        0  2504  1213   207
E1F4ELS33            402  457  4982  3615      756  2686  1254   417
E3F1ELS37            402  436     0  3816      8

In [52]:
import pickle
import numpy as np
import pandas as pd

# Load the data
TRAINING_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi.pkl"

with open(TRAINING_DATA_FILE, "rb") as f:
    full_dict = pickle.load(f)

# Define the mice IDs to keep
c_mice_ids = ['C6ELS9', 'C7ELS11', 'C2F4ELS18', 'C5F4ELS20', 'C7F1ELS22', 
              'C1F3ELS32', 'C5F3ELS40', 'C6F5ELS42', 'C7F2ELS45']
e_mice_ids = ['E2ELS3', 'E1F4ELS33', 'E3F1ELS37', 'E4F2ELS39', 'E5F1ELS41', 'E6F4ELS44']
mice_to_keep = c_mice_ids + e_mice_ids

print(f"Total samples in original data: {len(full_dict['mouse_id'])}")
print(f"Unique mice in original data: {len(np.unique(full_dict['mouse_id']))}")

def filter_and_trim_data(full_dict, mice_to_keep, trim_criteria):
    """
    First filter to keep only specified mice, then trim data based on criteria.
    
    Parameters:
    - full_dict: dictionary containing all data
    - mice_to_keep: list of short mouse IDs to keep (without 'Mouse' prefix)
    - trim_criteria: dictionary mapping stage -> max_samples (or None to keep all)
    
    Returns:
    - trimmed_dict: dictionary with filtered and trimmed data
    """
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame({
        'mouse_id': full_dict['mouse_id'],
        'period': full_dict['period'],
        'index': np.arange(len(full_dict['mouse_id']))
    })
    
    # Add short mouse ID
    df['mouse_id_short'] = df['mouse_id'].str.replace('Mouse', '')
    
    # STEP 1: Filter to keep only specified mice
    df = df[df['mouse_id_short'].isin(mice_to_keep)].copy()
    
    print(f"\nAfter filtering to keep only specified mice:")
    print(f"  Total samples: {len(df)}")
    print(f"  Unique mice: {df['mouse_id_short'].nunique()}")
    
    # STEP 2: Apply trim criteria
    print(f"\nApplying trim criteria:")
    for stage in sorted(trim_criteria.keys()):
        max_samples = trim_criteria[stage]
        if max_samples is not None:
            print(f"  {stage}: first {max_samples} samples per mouse")
        else:
            print(f"  {stage}: keep all samples")
    
    # Group by mouse and period, then take first N samples for each group
    indices_to_keep = []
    
    for (mouse_id, period), group in df.groupby(['mouse_id', 'period']):
        max_samples = trim_criteria.get(period, None)
        
        if max_samples is not None:
            # Take first N samples
            selected_indices = group['index'].values[:max_samples]
        else:
            # Keep all samples
            selected_indices = group['index'].values
        
        indices_to_keep.extend(selected_indices)
    
    indices_to_keep = np.array(sorted(indices_to_keep))
    
    print(f"\nAfter trimming:")
    print(f"  Total samples: {len(indices_to_keep)}")
    
    # Create trimmed dictionary
    trimmed_dict = {}
    for key in full_dict.keys():
        if key in ['freq_band', 'region', 'region_pair']:
            # These are metadata, keep as is
            trimmed_dict[key] = full_dict[key]
        else:
            # These are arrays that need to be indexed
            trimmed_dict[key] = full_dict[key][indices_to_keep]
    
    return trimmed_dict

# Calculate samples based on time windows (3 seconds per sample)
samples_per_minute = 20  # 60 seconds / 3 seconds per sample
samples_per_hour = samples_per_minute * 60  # 1200 samples per hour

# ============================================================================
# TRIM CRITERIA 1: Post stages only
# ============================================================================
print("\n" + "="*80)
print("PROCESSING TRIM CRITERIA 1: Post stages only")
print("="*80)

trim_criteria_1 = {
    'P1': 4 * samples_per_hour,      # 4 hours = 4800 samples
    'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
    'P8': 2 * samples_per_hour,      # 2 hours = 2400 samples
    'P14': 1 * samples_per_hour,     # 1 hour = 1200 samples
    'P4 home': 20 * samples_per_minute,  # 20 mins = 400 samples
    # Other stages keep all samples
    'Pre home': None,
    'Ges': None,
    'P20': None
}

trimmed_dict_1 = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria_1)

# Save Trim Criteria 1
output_file_1 = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi_Trim_Post.pkl"
with open(output_file_1, "wb") as f:
    pickle.dump(trimmed_dict_1, f)
print(f"\n✓ Saved Trim Criteria 1 to: {output_file_1}")

# Print summary for Trim Criteria 1
print("\nSummary statistics for Trim Criteria 1:")
df1 = pd.DataFrame({
    'mouse_id_short': [mid.replace('Mouse', '') for mid in trimmed_dict_1['mouse_id']],
    'period': trimmed_dict_1['period']
})
crosstab1 = pd.crosstab(df1['mouse_id_short'], df1['period'])
stage_order = ['Pre home', 'Ges', 'P1', 'P3', 'P4 home', 'P8', 'P14', 'P20']
existing_stages = [stage for stage in stage_order if stage in crosstab1.columns]
crosstab1 = crosstab1[existing_stages]
existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab1.index]
crosstab1 = crosstab1.loc[existing_mice]
print(crosstab1)

# ============================================================================
# TRIM CRITERIA 2: All stages with specific limits
# ============================================================================
print("\n" + "="*80)
print("PROCESSING TRIM CRITERIA 2: All stages with limits")
print("="*80)

trim_criteria_2 = {
    'P1': 4 * samples_per_hour,      # 4 hours = 4800 samples
    'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
    'P8': 2 * samples_per_hour,      # 2 hours = 2400 samples
    'P14': 1 * samples_per_hour,     # 1 hour = 1200 samples
    'P4 home': 20 * samples_per_minute,  # 20 mins = 400 samples
    'Pre home': 10 * samples_per_minute,  # 10 mins = 200 samples
    'Ges': 10 * samples_per_minute,  # 10 mins = 200 samples
    'P20': 10 * samples_per_minute   # 10 mins = 200 samples
}

trimmed_dict_2 = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria_2)

# Save Trim Criteria 2
output_file_2 = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi_Trim_All.pkl"
with open(output_file_2, "wb") as f:
    pickle.dump(trimmed_dict_2, f)
print(f"\n✓ Saved Trim Criteria 2 to: {output_file_2}")

# Print summary for Trim Criteria 2
print("\nSummary statistics for Trim Criteria 2:")
df2 = pd.DataFrame({
    'mouse_id_short': [mid.replace('Mouse', '') for mid in trimmed_dict_2['mouse_id']],
    'period': trimmed_dict_2['period']
})
crosstab2 = pd.crosstab(df2['mouse_id_short'], df2['period'])
crosstab2 = crosstab2[existing_stages]
existing_mice_2 = [mouse for mouse in mice_to_keep if mouse in crosstab2.index]
crosstab2 = crosstab2.loc[existing_mice_2]
print(crosstab2)

print("\n" + "="*80)
print("PROCESSING COMPLETE!")
print("="*80)
print(f"\nOriginal data: {len(full_dict['mouse_id'])} samples")
print(f"Trim Criteria 1: {len(trimmed_dict_1['mouse_id'])} samples")
print(f"Trim Criteria 2: {len(trimmed_dict_2['mouse_id'])} samples")

Total samples in original data: 190145
Unique mice in original data: 18

PROCESSING TRIM CRITERIA 1: Post stages only

After filtering to keep only specified mice:
  Total samples: 189123
  Unique mice: 15

Applying trim criteria:
  Ges: keep all samples
  P1: first 4800 samples per mouse
  P14: first 1200 samples per mouse
  P20: keep all samples
  P3: first 3600 samples per mouse
  P4 home: first 400 samples per mouse
  P8: first 2400 samples per mouse
  Pre home: keep all samples

After trimming:
  Total samples: 172205

✓ Saved Trim Criteria 1 to: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combine/full_spec_features_8roi_Trim_Post.pkl

Summary statistics for Trim Criteria 1:
period          Pre home  Ges    P1    P3  P4 home    P8   P14   P20
mouse_id_short                                                      
C6ELS9               203  212     0     0        0     0     0   213
C7ELS11              204  259     0  3600        0  2400  1200   212
C

### 2. Backproject on On/Off-nest

In [58]:
# Define the mice IDs to keep (matching the actual format in data: C1_ELS32 not C1F3ELS32)
c_mice_ids = ['C1_ELS32', 'C2_ELS18', 'C5_ELS40', 'C5_ELS20', 
              'C6_ELS42', 'C7_ELS11', 'C7_ELS22', 'C7_ELS45']
e_mice_ids = ['E1_ELS33', 'E2_ELS3', 'E3_ELS37', 'E4_ELS39', 
              'E5_ELS41', 'E6_ELS44']

# Combine the mice IDs
mice_to_keep = c_mice_ids + e_mice_ids

# Create DataFrame
df = pd.DataFrame({
    'mouse_id': full_dict['mouse_id'],
    'period': full_dict['period']
})

# Filter to keep only specified mice (no need to remove prefix since data already has the right format)
df_filtered = df[df['mouse_id'].isin(mice_to_keep)]

# Define the desired order for periods/stages
stage_order = ['P1', 'P3', 'P8']

# Create crosstab
crosstab = pd.crosstab(df_filtered['mouse_id'], df_filtered['period'])

# Reorder columns according to stage_order
crosstab = crosstab[stage_order]

# Reorder rows to match the order in c_mice_ids and e_mice_ids
existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab.index]
crosstab = crosstab.loc[existing_mice]

print("Sample counts for each mouse_id and stage combination:")
print("=" * 80)
print(crosstab)

Sample counts for each mouse_id and stage combination:
period      P1    P3    P8
mouse_id                  
C1_ELS32  4826  3756  2821
C2_ELS18  4908  3642  3178
C5_ELS40  5110  3625  2763
C5_ELS20  5009  3692  2524
C6_ELS42  5289  4308  2610
C7_ELS11     0  3604  3356
C7_ELS22  4862  4801  2489
C7_ELS45  4838  3695  2443
E1_ELS33  4982  3615  2686
E2_ELS3      0  3785  2504
E3_ELS37     0  3816  2438
E4_ELS39  4990  3690  2434
E5_ELS41  4876  4428  2469
E6_ELS44  5486  4806  2610


In [59]:
import pickle
import numpy as np
import pandas as pd

# Load the data
FULL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi.pkl"

with open(FULL_DATA_FILE, "rb") as f:
    full_dict = pickle.load(f)

# Define the mice IDs to keep (matching the actual format in data)
c_mice_ids = ['C1_ELS32', 'C2_ELS18', 'C5_ELS40', 'C5_ELS20', 
              'C6_ELS42', 'C7_ELS11', 'C7_ELS22', 'C7_ELS45']
e_mice_ids = ['E1_ELS33', 'E2_ELS3', 'E3_ELS37', 'E4_ELS39', 
              'E5_ELS41', 'E6_ELS44']
mice_to_keep = c_mice_ids + e_mice_ids

print(f"Total samples in original data: {len(full_dict['mouse_id'])}")
print(f"Unique mice in original data: {len(np.unique(full_dict['mouse_id']))}")

def filter_and_trim_data(full_dict, mice_to_keep, trim_criteria):
    """
    First filter to keep only specified mice, then trim data based on criteria.
    
    Parameters:
    - full_dict: dictionary containing all data
    - mice_to_keep: list of mouse IDs to keep (exact format as in data)
    - trim_criteria: dictionary mapping stage -> max_samples
    
    Returns:
    - trimmed_dict: dictionary with filtered and trimmed data
    """
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame({
        'mouse_id': full_dict['mouse_id'],
        'period': full_dict['period'],
        'index': np.arange(len(full_dict['mouse_id']))
    })
    
    # STEP 1: Filter to keep only specified mice
    df = df[df['mouse_id'].isin(mice_to_keep)].copy()
    
    print(f"\nAfter filtering to keep only specified mice:")
    print(f"  Total samples: {len(df)}")
    print(f"  Unique mice: {df['mouse_id'].nunique()}")
    
    # STEP 2: Apply trim criteria
    print(f"\nApplying trim criteria:")
    for stage in sorted(trim_criteria.keys()):
        max_samples = trim_criteria[stage]
        print(f"  {stage}: first {max_samples} samples per mouse")
    
    # Group by mouse and period, then take first N samples for each group
    indices_to_keep = []
    
    for (mouse_id, period), group in df.groupby(['mouse_id', 'period']):
        max_samples = trim_criteria.get(period)
        
        if max_samples is not None:
            # Take first N samples
            selected_indices = group['index'].values[:max_samples]
        else:
            # This shouldn't happen, but keep all if no criteria specified
            selected_indices = group['index'].values
        
        indices_to_keep.extend(selected_indices)
    
    indices_to_keep = np.array(sorted(indices_to_keep))
    
    print(f"\nAfter trimming:")
    print(f"  Total samples: {len(indices_to_keep)}")
    
    # Create trimmed dictionary
    trimmed_dict = {}
    for key in full_dict.keys():
        if key in ['freq_band', 'region', 'region_pair']:
            # These are metadata, keep as is
            trimmed_dict[key] = full_dict[key]
        else:
            # These are arrays that need to be indexed
            trimmed_dict[key] = full_dict[key][indices_to_keep]
    
    return trimmed_dict

# Calculate samples based on time windows (3 seconds per sample)
samples_per_minute = 20  # 60 seconds / 3 seconds per sample
samples_per_hour = samples_per_minute * 60  # 1200 samples per hour

# ============================================================================
# TRIM ON-NEST DATA: P1/3/8 only
# ============================================================================
print("\n" + "="*80)
print("PROCESSING ON-NEST DATA TRIMMING")
print("="*80)

trim_criteria = {
    'P1': 4 * samples_per_hour,      # 4 hours = 4800 samples
    'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
    'P8': 2 * samples_per_hour,      # 2 hours = 2400 samples
}

trimmed_dict = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria)

# Save trimmed data
output_file = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Trim.pkl"
with open(output_file, "wb") as f:
    pickle.dump(trimmed_dict, f)
print(f"\n✓ Saved trimmed on-nest data to: {output_file}")

# Print summary statistics
print("\nSummary statistics after trimming:")
df_trimmed = pd.DataFrame({
    'mouse_id': trimmed_dict['mouse_id'],
    'period': trimmed_dict['period']
})
crosstab = pd.crosstab(df_trimmed['mouse_id'], df_trimmed['period'])
stage_order = ['P1', 'P3', 'P8']
crosstab = crosstab[stage_order]
existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab.index]
crosstab = crosstab.loc[existing_mice]
print(crosstab)

# Print additional info about labels
print("\nLabel distribution (onnest_label):")
print(pd.Series(trimmed_dict['onnest_label']).value_counts().sort_index())
print("\nLabel distribution (onnest_label2):")
print(pd.Series(trimmed_dict['onnest_label2']).value_counts().sort_index())

print("\n" + "="*80)
print("PROCESSING COMPLETE!")
print("="*80)
print(f"\nOriginal data: {len(full_dict['mouse_id'])} samples")
print(f"Trimmed data: {len(trimmed_dict['mouse_id'])} samples")
print(f"Reduction: {len(full_dict['mouse_id']) - len(trimmed_dict['mouse_id'])} samples removed")

Total samples in original data: 147764
Unique mice in original data: 14

PROCESSING ON-NEST DATA TRIMMING

After filtering to keep only specified mice:
  Total samples: 147764
  Unique mice: 14

Applying trim criteria:
  P1: first 4800 samples per mouse
  P3: first 3600 samples per mouse
  P8: first 2400 samples per mouse

After trimming:
  Total samples: 136800

✓ Saved trimmed on-nest data to: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Trim.pkl

Summary statistics after trimming:
period      P1    P3    P8
mouse_id                  
C1_ELS32  4800  3600  2400
C2_ELS18  4800  3600  2400
C5_ELS40  4800  3600  2400
C5_ELS20  4800  3600  2400
C6_ELS42  4800  3600  2400
C7_ELS11     0  3600  2400
C7_ELS22  4800  3600  2400
C7_ELS45  4800  3600  2400
E1_ELS33  4800  3600  2400
E2_ELS3      0  3600  2400
E3_ELS37     0  3600  2400
E4_ELS39  4800  3600  2400
E5_ELS41  4800  3600  2400
E6_ELS44  4800  3600  2400

Label

### 3. Backproject on Onnest Licking vs Onnest NonLicking

In [41]:
# Define the mice IDs to keep (matching the actual format: MouseC1F3ELS32)
c_mice_ids = ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 
              'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45']
e_mice_ids = ['MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 
              'MouseE5F1ELS41', 'MouseE6F4ELS44']

# Combine the mice IDs
mice_to_keep = c_mice_ids + e_mice_ids

# Create DataFrame
df = pd.DataFrame({
    'mouse_id': full_dict['mouse_id'],
    'period': full_dict['period']
})

# Filter to keep only specified mice
df_filtered = df[df['mouse_id'].isin(mice_to_keep)]

# Check if filtering worked
print(f"Total samples before filtering: {len(df)}")
print(f"Total samples after filtering: {len(df_filtered)}")
print(f"Unique mice after filtering: {df_filtered['mouse_id'].unique()}")

# Define the desired order for periods/stages
stage_order = ['P3']

# Create crosstab
crosstab = pd.crosstab(df_filtered['mouse_id'], df_filtered['period'])

# Remove 'Mouse' prefix for display
crosstab.index = crosstab.index.str.replace('Mouse', '')

print("\nSample counts for each mouse_id and stage combination:")
print("=" * 80)
print(crosstab)

Total samples before filtering: 119281
Total samples after filtering: 55263
Unique mice after filtering: ['MouseC1F3ELS32' 'MouseC2F4ELS18' 'MouseC5F3ELS40' 'MouseC5F4ELS20'
 'MouseC6F5ELS42' 'MouseC7ELS11' 'MouseC7F1ELS22' 'MouseC7F2ELS45'
 'MouseE1F4ELS33' 'MouseE2ELS3' 'MouseE3F1ELS37' 'MouseE4F2ELS39'
 'MouseE5F1ELS41' 'MouseE6F4ELS44']

Sample counts for each mouse_id and stage combination:
period       P3
mouse_id       
C1F3ELS32  3756
C2F4ELS18  3642
C5F3ELS40  3625
C5F4ELS20  3692
C6F5ELS42  4308
C7ELS11    3604
C7F1ELS22  4801
C7F2ELS45  3695
E1F4ELS33  3615
E2ELS3     3785
E3F1ELS37  3816
E4F2ELS39  3690
E5F1ELS41  4428
E6F4ELS44  4806


In [60]:
import pickle
import numpy as np
import pandas as pd

# Load the data
FULL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick/full_onnest_lick.pkl"

with open(FULL_DATA_FILE, "rb") as f:
    full_dict = pickle.load(f)

# Define the mice IDs to keep (matching the actual format: MouseC1F3ELS32)
c_mice_ids = ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 
              'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45']
e_mice_ids = ['MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 
              'MouseE5F1ELS41', 'MouseE6F4ELS44']
mice_to_keep = c_mice_ids + e_mice_ids

print(f"Total samples in original data: {len(full_dict['mouse_id'])}")
print(f"Unique mice in original data: {len(np.unique(full_dict['mouse_id']))}")
print(f"All unique mice: {np.unique(full_dict['mouse_id'])}")

def filter_and_trim_data(full_dict, mice_to_keep, trim_criteria):
    """
    First filter to keep only specified mice, then trim data based on criteria.
    
    Parameters:
    - full_dict: dictionary containing all data
    - mice_to_keep: list of mouse IDs to keep (exact format as in data)
    - trim_criteria: dictionary mapping stage -> max_samples
    
    Returns:
    - trimmed_dict: dictionary with filtered and trimmed data
    """
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame({
        'mouse_id': full_dict['mouse_id'],
        'period': full_dict['period'],
        'index': np.arange(len(full_dict['mouse_id']))
    })
    
    # STEP 1: Filter to keep only specified mice
    df = df[df['mouse_id'].isin(mice_to_keep)].copy()
    
    print(f"\nAfter filtering to keep only specified mice:")
    print(f"  Total samples: {len(df)}")
    print(f"  Unique mice: {df['mouse_id'].nunique()}")
    print(f"  Mice kept: {sorted(df['mouse_id'].unique())}")
    
    # STEP 2: Apply trim criteria
    print(f"\nApplying trim criteria:")
    for stage in sorted(trim_criteria.keys()):
        max_samples = trim_criteria[stage]
        print(f"  {stage}: first {max_samples} samples per mouse")
    
    # Group by mouse and period, then take first N samples for each group
    indices_to_keep = []
    
    for (mouse_id, period), group in df.groupby(['mouse_id', 'period']):
        max_samples = trim_criteria.get(period)
        
        if max_samples is not None:
            # Take first N samples
            selected_indices = group['index'].values[:max_samples]
        else:
            # This shouldn't happen, but keep all if no criteria specified
            selected_indices = group['index'].values
        
        indices_to_keep.extend(selected_indices)
    
    indices_to_keep = np.array(sorted(indices_to_keep))
    
    print(f"\nAfter trimming:")
    print(f"  Total samples: {len(indices_to_keep)}")
    
    # Create trimmed dictionary
    trimmed_dict = {}
    for key in full_dict.keys():
        if key in ['freq_band', 'region', 'region_pair']:
            # These are metadata, keep as is
            trimmed_dict[key] = full_dict[key]
        else:
            # These are arrays that need to be indexed
            trimmed_dict[key] = full_dict[key][indices_to_keep]
    
    return trimmed_dict

# Calculate samples based on time windows (3 seconds per sample)
samples_per_minute = 20  # 60 seconds / 3 seconds per sample
samples_per_hour = samples_per_minute * 60  # 1200 samples per hour

# ============================================================================
# TRIM P3 ON-NEST LICKING DATA
# ============================================================================
print("\n" + "="*80)
print("PROCESSING P3 ON-NEST LICKING DATA TRIMMING")
print("="*80)

trim_criteria = {
    'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
}

trimmed_dict = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria)

# Save trimmed data
output_file = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick/full_onnest_lick_Trim.pkl"
with open(output_file, "wb") as f:
    pickle.dump(trimmed_dict, f)
print(f"\n✓ Saved trimmed P3 on-nest licking data to: {output_file}")

# Print summary statistics
print("\nSummary statistics after trimming:")
df_trimmed = pd.DataFrame({
    'mouse_id': trimmed_dict['mouse_id'],
    'period': trimmed_dict['period']
})

# Count samples per mouse
mouse_counts = df_trimmed['mouse_id'].value_counts().sort_index()
print("\nSamples per mouse:")
for mouse_id in mice_to_keep:
    if mouse_id in mouse_counts.index:
        print(f"  {mouse_id}: {mouse_counts[mouse_id]} samples")
    else:
        print(f"  {mouse_id}: 0 samples (not in data)")

# Print label distributions
print("\nLabel distribution (onnest_raw):")
onnest_dist = pd.Series(trimmed_dict['onnest_raw']).value_counts().sort_index()
print(onnest_dist)
print(f"  Total: {len(trimmed_dict['onnest_raw'])} samples")

print("\nLabel distribution (licking):")
licking_dist = pd.Series(trimmed_dict['licking']).value_counts().sort_index()
print(licking_dist)
print(f"  Total: {len(trimmed_dict['licking'])} samples")

# Cross-tabulation of onnest_raw and licking
print("\nCross-tabulation (onnest_raw vs licking):")
crosstab_labels = pd.crosstab(
    trimmed_dict['onnest_raw'], 
    trimmed_dict['licking'],
    rownames=['onnest_raw'],
    colnames=['licking']
)
print(crosstab_labels)

print("\n" + "="*80)
print("PROCESSING COMPLETE!")
print("="*80)
print(f"\nOriginal data: {len(full_dict['mouse_id'])} samples")
print(f"Trimmed data: {len(trimmed_dict['mouse_id'])} samples")
print(f"Reduction: {len(full_dict['mouse_id']) - len(trimmed_dict['mouse_id'])} samples removed")

Total samples in original data: 119281
Unique mice in original data: 30
All unique mice: ['MouseC10F4ELS26' 'MouseC11F1ELS29' 'MouseC1ELS16' 'MouseC1F3ELS32'
 'MouseC2F4ELS18' 'MouseC3F3ELS36' 'MouseC4F4ELS38' 'MouseC5F3ELS40'
 'MouseC5F4ELS20' 'MouseC6F5ELS42' 'MouseC7ELS11' 'MouseC7F1ELS22'
 'MouseC7F2ELS45' 'MouseC7F4ELS12' 'MouseC8ELS13' 'MouseC9F3ELS24'
 'MouseE11F2ELS28' 'MouseE12F4ELS31' 'MouseE1F4ELS33' 'MouseE2ELS3'
 'MouseE3ELS7' 'MouseE3F1ELS37' 'MouseE4F2ELS39' 'MouseE4F3ELS19'
 'MouseE5ELS8' 'MouseE5F1ELS41' 'MouseE6F4ELS44' 'MouseE7F2ELS21'
 'MouseE8F4ELS23' 'MouseE9ELS15']

PROCESSING P3 ON-NEST LICKING DATA TRIMMING

After filtering to keep only specified mice:
  Total samples: 55263
  Unique mice: 14
  Mice kept: ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45', 'MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 'MouseE5F1ELS41', 'MouseE6F4ELS44']

Applying tr

### 4. Backproject on Onnest Licking vs Grooming

In [48]:
# Define the mice IDs to keep (matching the actual format: MouseC1F3ELS32)
c_mice_ids = ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 
              'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45']
e_mice_ids = ['MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 
              'MouseE5F1ELS41', 'MouseE6F4ELS44']

# Combine the mice IDs
mice_to_keep = c_mice_ids + e_mice_ids

# Create DataFrame
df = pd.DataFrame({
    'mouse_id': full_dict['mouse_id'],
    'period': full_dict['period']
})

# Filter to keep only specified mice
df_filtered = df[df['mouse_id'].isin(mice_to_keep)]

# Check if filtering worked
print(f"Total samples before filtering: {len(df)}")
print(f"Total samples after filtering: {len(df_filtered)}")
print(f"Unique mice after filtering: {df_filtered['mouse_id'].unique()}")

# Define the desired order for periods/stages
stage_order = ['P3']

# Create crosstab
crosstab = pd.crosstab(df_filtered['mouse_id'], df_filtered['period'])

# Remove 'Mouse' prefix for display
crosstab.index = crosstab.index.str.replace('Mouse', '')

print("\nSample counts for each mouse_id and stage combination:")
print("=" * 80)
print(crosstab)

Total samples before filtering: 119281
Total samples after filtering: 55263
Unique mice after filtering: ['MouseC1F3ELS32' 'MouseC2F4ELS18' 'MouseC5F3ELS40' 'MouseC5F4ELS20'
 'MouseC6F5ELS42' 'MouseC7ELS11' 'MouseC7F1ELS22' 'MouseC7F2ELS45'
 'MouseE1F4ELS33' 'MouseE2ELS3' 'MouseE3F1ELS37' 'MouseE4F2ELS39'
 'MouseE5F1ELS41' 'MouseE6F4ELS44']

Sample counts for each mouse_id and stage combination:
period       P3
mouse_id       
C1F3ELS32  3756
C2F4ELS18  3642
C5F3ELS40  3625
C5F4ELS20  3692
C6F5ELS42  4308
C7ELS11    3604
C7F1ELS22  4801
C7F2ELS45  3695
E1F4ELS33  3615
E2ELS3     3785
E3F1ELS37  3816
E4F2ELS39  3690
E5F1ELS41  4428
E6F4ELS44  4806


In [61]:
import pickle
import numpy as np
import pandas as pd

# Load the data
FULL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field.pkl"

with open(FULL_DATA_FILE, "rb") as f:
    full_dict = pickle.load(f)

# Define the mice IDs to keep (matching the actual format: MouseC1F3ELS32)
c_mice_ids = ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 
              'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45']
e_mice_ids = ['MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 
              'MouseE5F1ELS41', 'MouseE6F4ELS44']
mice_to_keep = c_mice_ids + e_mice_ids

print(f"Total samples in original data: {len(full_dict['mouse_id'])}")
print(f"Unique mice in original data: {len(np.unique(full_dict['mouse_id']))}")
print(f"All unique mice: {np.unique(full_dict['mouse_id'])}")

def filter_and_trim_data(full_dict, mice_to_keep, trim_criteria):
    """
    First filter to keep only specified mice, then trim data based on criteria.
    
    Parameters:
    - full_dict: dictionary containing all data
    - mice_to_keep: list of mouse IDs to keep (exact format as in data)
    - trim_criteria: dictionary mapping stage -> max_samples
    
    Returns:
    - trimmed_dict: dictionary with filtered and trimmed data
    """
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame({
        'mouse_id': full_dict['mouse_id'],
        'period': full_dict['period'],
        'index': np.arange(len(full_dict['mouse_id']))
    })
    
    # STEP 1: Filter to keep only specified mice
    df = df[df['mouse_id'].isin(mice_to_keep)].copy()
    
    print(f"\nAfter filtering to keep only specified mice:")
    print(f"  Total samples: {len(df)}")
    print(f"  Unique mice: {df['mouse_id'].nunique()}")
    print(f"  Mice kept: {sorted(df['mouse_id'].unique())}")
    
    # STEP 2: Apply trim criteria
    print(f"\nApplying trim criteria:")
    for stage in sorted(trim_criteria.keys()):
        max_samples = trim_criteria[stage]
        print(f"  {stage}: first {max_samples} samples per mouse")
    
    # Group by mouse and period, then take first N samples for each group
    indices_to_keep = []
    
    for (mouse_id, period), group in df.groupby(['mouse_id', 'period']):
        max_samples = trim_criteria.get(period)
        
        if max_samples is not None:
            # Take first N samples
            selected_indices = group['index'].values[:max_samples]
        else:
            # This shouldn't happen, but keep all if no criteria specified
            selected_indices = group['index'].values
        
        indices_to_keep.extend(selected_indices)
    
    indices_to_keep = np.array(sorted(indices_to_keep))
    
    print(f"\nAfter trimming:")
    print(f"  Total samples: {len(indices_to_keep)}")
    
    # Create trimmed dictionary
    trimmed_dict = {}
    for key in full_dict.keys():
        if key in ['freq_band', 'region', 'region_pair']:
            # These are metadata, keep as is
            trimmed_dict[key] = full_dict[key]
        else:
            # These are arrays that need to be indexed
            trimmed_dict[key] = full_dict[key][indices_to_keep]
    
    return trimmed_dict

# Calculate samples based on time windows (3 seconds per sample)
samples_per_minute = 20  # 60 seconds / 3 seconds per sample
samples_per_hour = samples_per_minute * 60  # 1200 samples per hour

# ============================================================================
# TRIM P3 MULTIPLE BEHAVIORS DATA
# ============================================================================
print("\n" + "="*80)
print("PROCESSING P3 MULTIPLE BEHAVIORS DATA TRIMMING")
print("="*80)

trim_criteria = {
    'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
}

trimmed_dict = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria)

# Save trimmed data
output_file = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field_Trim.pkl"
with open(output_file, "wb") as f:
    pickle.dump(trimmed_dict, f)
print(f"\n✓ Saved trimmed P3 multiple behaviors data to: {output_file}")

# Print summary statistics
print("\nSummary statistics after trimming:")
df_trimmed = pd.DataFrame({
    'mouse_id': trimmed_dict['mouse_id'],
    'period': trimmed_dict['period']
})

# Count samples per mouse
mouse_counts = df_trimmed['mouse_id'].value_counts().sort_index()
print("\nSamples per mouse:")
for mouse_id in mice_to_keep:
    if mouse_id in mouse_counts.index:
        print(f"  {mouse_id}: {mouse_counts[mouse_id]} samples")
    else:
        print(f"  {mouse_id}: 0 samples (not in data)")

# Print label distributions for all behavior labels
print("\n" + "="*80)
print("BEHAVIOR LABEL DISTRIBUTIONS")
print("="*80)

print("\nLabel distribution (onnest_raw):")
onnest_dist = pd.Series(trimmed_dict['onnest_raw']).value_counts().sort_index()
print(onnest_dist)
print(f"  Percentage on nest: {100 * onnest_dist.get(1, 0) / len(trimmed_dict['onnest_raw']):.2f}%")

print("\nLabel distribution (licking):")
licking_dist = pd.Series(trimmed_dict['licking']).value_counts().sort_index()
print(licking_dist)
print(f"  Percentage licking: {100 * licking_dist.get(1, 0) / len(trimmed_dict['licking']):.2f}%")

print("\nLabel distribution (selfgroom):")
selfgroom_dist = pd.Series(trimmed_dict['selfgroom']).value_counts().sort_index()
print(selfgroom_dist)
print(f"  Percentage self-grooming: {100 * selfgroom_dist.get(1, 0) / len(trimmed_dict['selfgroom']):.2f}%")

# Cross-tabulation of behaviors
print("\n" + "="*80)
print("BEHAVIOR CO-OCCURRENCE")
print("="*80)

print("\nOnnest_raw vs Licking:")
crosstab_onnest_lick = pd.crosstab(
    trimmed_dict['onnest_raw'], 
    trimmed_dict['licking'],
    rownames=['onnest_raw'],
    colnames=['licking']
)
print(crosstab_onnest_lick)

print("\nOnnest_raw vs Selfgroom:")
crosstab_onnest_groom = pd.crosstab(
    trimmed_dict['onnest_raw'], 
    trimmed_dict['selfgroom'],
    rownames=['onnest_raw'],
    colnames=['selfgroom']
)
print(crosstab_onnest_groom)

print("\nLicking vs Selfgroom:")
crosstab_lick_groom = pd.crosstab(
    trimmed_dict['licking'], 
    trimmed_dict['selfgroom'],
    rownames=['licking'],
    colnames=['selfgroom']
)
print(crosstab_lick_groom)

# 3-way behavior combinations
print("\nBehavior combinations (onnest, licking, selfgroom):")
behavior_combo = pd.DataFrame({
    'onnest': trimmed_dict['onnest_raw'],
    'licking': trimmed_dict['licking'],
    'selfgroom': trimmed_dict['selfgroom']
})
combo_counts = behavior_combo.value_counts().sort_index()
print(combo_counts)

print("\n" + "="*80)
print("PROCESSING COMPLETE!")
print("="*80)
print(f"\nOriginal data: {len(full_dict['mouse_id'])} samples")
print(f"Trimmed data: {len(trimmed_dict['mouse_id'])} samples")
print(f"Reduction: {len(full_dict['mouse_id']) - len(trimmed_dict['mouse_id'])} samples removed")

Total samples in original data: 119281
Unique mice in original data: 30
All unique mice: ['MouseC10F4ELS26' 'MouseC11F1ELS29' 'MouseC1ELS16' 'MouseC1F3ELS32'
 'MouseC2F4ELS18' 'MouseC3F3ELS36' 'MouseC4F4ELS38' 'MouseC5F3ELS40'
 'MouseC5F4ELS20' 'MouseC6F5ELS42' 'MouseC7ELS11' 'MouseC7F1ELS22'
 'MouseC7F2ELS45' 'MouseC7F4ELS12' 'MouseC8ELS13' 'MouseC9F3ELS24'
 'MouseE11F2ELS28' 'MouseE12F4ELS31' 'MouseE1F4ELS33' 'MouseE2ELS3'
 'MouseE3ELS7' 'MouseE3F1ELS37' 'MouseE4F2ELS39' 'MouseE4F3ELS19'
 'MouseE5ELS8' 'MouseE5F1ELS41' 'MouseE6F4ELS44' 'MouseE7F2ELS21'
 'MouseE8F4ELS23' 'MouseE9ELS15']

PROCESSING P3 MULTIPLE BEHAVIORS DATA TRIMMING

After filtering to keep only specified mice:
  Total samples: 55263
  Unique mice: 14
  Mice kept: ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45', 'MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 'MouseE5F1ELS41', 'MouseE6F4ELS44']

Applying

# 10. Jan 06 -- Sara's request on 3 files

In [5]:
import pickle

# Load the pickle file
file_path = '/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_All/Combined/full_onnest_spec_features.pkl'

with open(file_path, 'rb') as f:
    data = pickle.load(f)

print(f"Data type: {type(data)}")

# If it's a dictionary, show keys and sample values
if isinstance(data, dict):
    print(f"Number of keys: {len(data)}")
    print("\nAll keys:")
    for key in data.keys():
        print(f"  {key}")
    
    print("\nSample content for each key:")
    for key in data.keys():
        print(f"\nKey: {key}")
        print(f"Type: {type(data[key])}")
        
        if hasattr(data[key], '__len__'):
            print(f"Length: {len(data[key])}")
        
        # Show first few elements
        if isinstance(data[key], (list, tuple)):
            print(f"First 5 elements: {data[key][:5]}")
        elif isinstance(data[key], dict):
            sample_keys = list(data[key].keys())[:5]
            print(f"Sample keys: {sample_keys}")
        elif hasattr(data[key], 'shape'):  # numpy array
            print(f"Shape: {data[key].shape}")
            if len(data[key].shape) == 1:  # 1D array
                print(f"First 10 values: {data[key][:10]}")
                print(f"Unique values: {len(set(data[key]))} unique")
                print(f"Sample unique values: {list(set(data[key]))[:10]}")
            else:  # 2D or higher
                print(f"First 3 rows:")
                print(data[key][:3])
        else:
            print(f"Value: {data[key]}")

else:
    print(f"Data: {data}")
    if hasattr(data, '__len__'):
        print(f"Length: {len(data)}")

创建完整数据集（对齐nursing、筛选指定小鼠、trim到3600样本）

要保留的小鼠:
  C组: 8只 - ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45']
  E组: 6只 - ['MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 'MouseE5F1ELS41', 'MouseE6F4ELS44']
  总计: 14只

步骤1: 筛选指定的小鼠...
原始样本数: 119,281
筛选后样本数: 55,263

在筛选后的数据中:
  有nursing数据的小鼠: 12只
  没有nursing数据的小鼠: ['MouseE2ELS3', 'MouseE5F1ELS41']

步骤2: 按recording对齐nursing数据...
Recordings数量: 14

对齐完成:
  成功对齐的recordings: 12
  成功对齐的样本数: 47,050
  未对齐的recordings: 0

步骤3: Trim每个recording到前3600个样本...
  MouseC1F3ELS32 - P3: 3756 → 3600 (删除 156 个样本)
  MouseC2F4ELS18 - P3: 3642 → 3600 (删除 42 个样本)
  MouseC5F3ELS40 - P3: 3625 → 3600 (删除 25 个样本)
  MouseC5F4ELS20 - P3: 3692 → 3600 (删除 92 个样本)
  MouseC6F5ELS42 - P3: 4308 → 3600 (删除 708 个样本)
  MouseC7ELS11 - P3: 3604 → 3600 (删除 4 个样本)
  MouseC7F1ELS22 - P3: 4801 → 3600 (删除 1201 个样本)
  MouseC7F2ELS45 - P3: 3695 → 3600 (删除 95 个样本)
  MouseE1F4ELS

# 11. Jan12 onnest modification — fix E5ELS41 P1 video issue → generates `_Jan212026_Trim.pkl`

In [28]:
import numpy as np
import pickle
import re
import os

# Define target mice
target_mice = {
    'C1_ELS1', 'C6_ELS9', 'C7_ELS11', 'E2_ELS3', 'C2_ELS18', 
    'C5_ELS20', 'C7_ELS22', 'C1_ELS32', 'C5_ELS40', 'C6_ELS42', 
    'C7_ELS45', 'E1_ELS33', 'E2_ELS35', 'E3_ELS37', 'E4_ELS39', 
    'E5_ELS41', 'E6_ELS43', 'E6_ELS44'
}

# Function to convert mouse ID format
def convert_mouse_id(original_id):
    """
    Convert mouse ID to CX_ELSXX or EX_ELSXX format
    """
    # Pattern 1: MouseCXFXELSXX -> CX_ELSXX
    pattern1 = r'MouseC(\d+)F\d+ELS(\d+)'
    match1 = re.match(pattern1, original_id)
    if match1:
        return f"C{match1.group(1)}_ELS{match1.group(2)}"
    
    # Pattern 2: MouseCXELSXX -> CX_ELSXX
    pattern2 = r'MouseC(\d+)ELS(\d+)'
    match2 = re.match(pattern2, original_id)
    if match2:
        return f"C{match2.group(1)}_ELS{match2.group(2)}"
    
    # Pattern 3: MouseEXFXELSXX -> EX_ELSXX
    pattern3 = r'MouseE(\d+)F\d+ELS(\d+)'
    match3 = re.match(pattern3, original_id)
    if match3:
        return f"E{match3.group(1)}_ELS{match3.group(2)}"
    
    # Pattern 4: MouseEXELSXX -> EX_ELSXX
    pattern4 = r'MouseE(\d+)ELS(\d+)'
    match4 = re.match(pattern4, original_id)
    if match4:
        return f"E{match4.group(1)}_ELS{match4.group(2)}"
    
    # Pattern 5: Already in correct format CXELSXX -> CX_ELSXX
    pattern5 = r'C(\d+)ELS(\d+)'
    match5 = re.match(pattern5, original_id)
    if match5:
        return f"C{match5.group(1)}_ELS{match5.group(2)}"
    
    # Pattern 6: Already in correct format EXELSXX -> EX_ELSXX
    pattern6 = r'E(\d+)ELS(\d+)'
    match6 = re.match(pattern6, original_id)
    if match6:
        return f"E{match6.group(1)}_ELS{match6.group(2)}"
    
    return original_id  # Return original if no pattern matches

# Convert all mouse IDs
print("Converting mouse IDs...")
converted_mouse_ids = np.array([convert_mouse_id(mid) for mid in data['mouse_id']])

# Check unique converted IDs
unique_converted = np.unique(converted_mouse_ids)
print(f"Number of unique converted mouse IDs: {len(unique_converted)}")
print(f"Converted mouse IDs: {unique_converted}")

# Find intersection with target mice
intersection = set(unique_converted) & target_mice
print(f"\nIntersection with target mice: {intersection}")
print(f"Number of mice in intersection: {len(intersection)}")

# Create mask for filtering
mask = np.isin(converted_mouse_ids, list(intersection))
print(f"Number of samples matching target mice: {np.sum(mask)}")

# Filter all data based on the mask
filtered_data = {}
for key in data.keys():
    if key in ['mouse_id', 'period', 'onnest_label']:
        # For 1D arrays, filter directly
        filtered_data[key] = data[key][mask]
    elif key in ['power', 'coh_sq_coherence']:
        # For 2D arrays, filter rows
        filtered_data[key] = data[key][mask]
    else:
        # For other keys (freq_band, region, region_pair), keep as is
        filtered_data[key] = data[key]

# Update mouse_id with converted format
filtered_data['mouse_id'] = converted_mouse_ids[mask]

# Print summary of filtered data
print(f"\nFiltered data summary:")
for key in filtered_data.keys():
    if hasattr(filtered_data[key], 'shape'):
        print(f"{key}: shape {filtered_data[key].shape}")
    else:
        print(f"{key}: {type(filtered_data[key])}, length {len(filtered_data[key])}")

# Check unique values in filtered data
print(f"\nUnique mouse IDs in filtered data: {np.unique(filtered_data['mouse_id'])}")

# Save filtered data
import os
file_path = '/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Jan212026.pkl'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'wb') as f:
    pickle.dump(filtered_data, f)

print(f"\nFiltered data saved to: {file_path}")

Converting mouse IDs...
Number of unique converted mouse IDs: 33
Converted mouse IDs: ['C10_ELS26' 'C11_ELS29' 'C11_ELS30' 'C1_ELS16' 'C1_ELS32' 'C2_ELS18'
 'C3_ELS36' 'C4_ELS38' 'C5_ELS20' 'C5_ELS40' 'C6_ELS42' 'C7_ELS11'
 'C7_ELS12' 'C7_ELS22' 'C7_ELS45' 'C8_ELS13' 'C9_ELS24' 'E10_ELS27'
 'E11_ELS28' 'E12_ELS31' 'E1_ELS33' 'E2_ELS3' 'E3_ELS37' 'E3_ELS7'
 'E4_ELS19' 'E4_ELS39' 'E5_ELS41' 'E5_ELS8' 'E6_ELS44' 'E7_ELS21'
 'E8_ELS14' 'E8_ELS23' 'E9_ELS15']

Intersection with target mice: {'C7_ELS11', 'C1_ELS32', 'C7_ELS45', 'C6_ELS42', 'E3_ELS37', 'C5_ELS40', 'C5_ELS20', 'C7_ELS22', 'E1_ELS33', 'E2_ELS3', 'E6_ELS44', 'E4_ELS39', 'E5_ELS41', 'C2_ELS18'}
Number of mice in intersection: 14
Number of samples matching target mice: 165210

Filtered data summary:
power: shape (165210, 24)
coh_sq_coherence: shape (165210, 84)
freq_band: <class 'list'>, length 3
region: <class 'list'>, length 24
region_pair: <class 'list'>, length 84
mouse_id: shape (165210,)
period: shape (165210,)
onnest_label:

In [30]:
# Define the mice IDs to keep (matching the actual format in data: C1_ELS32 not C1F3ELS32)
c_mice_ids = ['C1_ELS32', 'C2_ELS18', 'C5_ELS40', 'C5_ELS20', 
              'C6_ELS42', 'C7_ELS11', 'C7_ELS22', 'C7_ELS45']
e_mice_ids = ['E1_ELS33', 'E2_ELS3', 'E3_ELS37', 'E4_ELS39', 
              'E5_ELS41', 'E6_ELS44']

# Combine the mice IDs
mice_to_keep = c_mice_ids + e_mice_ids

# Create DataFrame
df = pd.DataFrame({
    'mouse_id': full_dict['mouse_id'],
    'period': full_dict['period']
})

# Filter to keep only specified mice (no need to remove prefix since data already has the right format)
df_filtered = df[df['mouse_id'].isin(mice_to_keep)]

# Define the desired order for periods/stages
stage_order = ['P1', 'P3', 'P8', 'P14']

# Create crosstab
crosstab = pd.crosstab(df_filtered['mouse_id'], df_filtered['period'])

# Reorder columns according to stage_order
crosstab = crosstab[stage_order]

# Reorder rows to match the order in c_mice_ids and e_mice_ids
existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab.index]
crosstab = crosstab.loc[existing_mice]

print("Sample counts for each mouse_id and stage combination:")
print("=" * 80)
print(crosstab)

Sample counts for each mouse_id and stage combination:
period      P1    P3    P8   P14
mouse_id                        
C1_ELS32  4826  3756  2821  1258
C2_ELS18  4908  3642  3178  1278
C5_ELS40  5110  3625  2763  1562
C5_ELS20  5009  3692  2524  1311
C6_ELS42  5289  4308  2610  1280
C7_ELS11     0  3604  3356  1304
C7_ELS22  4862  4801  2489  1354
C7_ELS45  4838  3695  2443  1797
E1_ELS33  4982  3615  2686  1254
E2_ELS3      0  3785  2504  1213
E3_ELS37     0  3816  2438  1266
E4_ELS39  4990  3690  2434  1293
E5_ELS41  4876  4428  2469     0
E6_ELS44  5486  4806  2610  1276


In [32]:
import pickle
import numpy as np
import pandas as pd

# Load the data
FULL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Jan212026.pkl"

with open(FULL_DATA_FILE, "rb") as f:
    full_dict = pickle.load(f)

# Define the mice IDs to keep (matching the actual format in data)
c_mice_ids = ['C1_ELS32', 'C2_ELS18', 'C5_ELS40', 'C5_ELS20', 
              'C6_ELS42', 'C7_ELS11', 'C7_ELS22', 'C7_ELS45']
e_mice_ids = ['E1_ELS33', 'E2_ELS3', 'E3_ELS37', 'E4_ELS39', 
              'E5_ELS41', 'E6_ELS44']
mice_to_keep = c_mice_ids + e_mice_ids

print(f"Total samples in original data: {len(full_dict['mouse_id'])}")
print(f"Unique mice in original data: {len(np.unique(full_dict['mouse_id']))}")

def filter_and_trim_data(full_dict, mice_to_keep, trim_criteria):
    """
    First filter to keep only specified mice, then trim data based on criteria.
    
    Parameters:
    - full_dict: dictionary containing all data
    - mice_to_keep: list of mouse IDs to keep (exact format as in data)
    - trim_criteria: dictionary mapping stage -> max_samples
    
    Returns:
    - trimmed_dict: dictionary with filtered and trimmed data
    """
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame({
        'mouse_id': full_dict['mouse_id'],
        'period': full_dict['period'],
        'index': np.arange(len(full_dict['mouse_id']))
    })
    
    # STEP 1: Filter to keep only specified mice
    df = df[df['mouse_id'].isin(mice_to_keep)].copy()
    
    print(f"\nAfter filtering to keep only specified mice:")
    print(f"  Total samples: {len(df)}")
    print(f"  Unique mice: {df['mouse_id'].nunique()}")
    
    # STEP 2: Apply trim criteria
    print(f"\nApplying trim criteria:")
    for stage in sorted(trim_criteria.keys()):
        max_samples = trim_criteria[stage]
        print(f"  {stage}: first {max_samples} samples per mouse")
    
    # Group by mouse and period, then take first N samples for each group
    indices_to_keep = []
    
    for (mouse_id, period), group in df.groupby(['mouse_id', 'period']):
        max_samples = trim_criteria.get(period)
        
        if max_samples is not None:
            # Take first N samples
            selected_indices = group['index'].values[:max_samples]
        else:
            # This shouldn't happen, but keep all if no criteria specified
            selected_indices = group['index'].values
        
        indices_to_keep.extend(selected_indices)
    
    indices_to_keep = np.array(sorted(indices_to_keep))
    
    print(f"\nAfter trimming:")
    print(f"  Total samples: {len(indices_to_keep)}")
    
    # Create trimmed dictionary
    trimmed_dict = {}
    for key in full_dict.keys():
        if key in ['freq_band', 'region', 'region_pair']:
            # These are metadata, keep as is
            trimmed_dict[key] = full_dict[key]
        else:
            # These are arrays that need to be indexed
            trimmed_dict[key] = full_dict[key][indices_to_keep]
    
    return trimmed_dict

# Calculate samples based on time windows (3 seconds per sample)
samples_per_minute = 20  # 60 seconds / 3 seconds per sample
samples_per_hour = samples_per_minute * 60  # 1200 samples per hour

# ============================================================================
# TRIM ON-NEST DATA: P1/3/8/14 only
# ============================================================================
print("\n" + "="*80)
print("PROCESSING ON-NEST DATA TRIMMING")
print("="*80)

trim_criteria = {
    'P1': 4 * samples_per_hour,      # 4 hours = 4800 samples
    'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
    'P8': 2 * samples_per_hour,      # 2 hours = 2400 samples
    'P14': 1 * samples_per_hour,      # 1 hours = 1200 samples    
}

trimmed_dict = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria)

# Save trimmed data
output_file = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Jan212026_Trim.pkl"
with open(output_file, "wb") as f:
    pickle.dump(trimmed_dict, f)
print(f"\n✓ Saved trimmed on-nest data to: {output_file}")

# Print summary statistics
print("\nSummary statistics after trimming:")
df_trimmed = pd.DataFrame({
    'mouse_id': trimmed_dict['mouse_id'],
    'period': trimmed_dict['period']
})
crosstab = pd.crosstab(df_trimmed['mouse_id'], df_trimmed['period'])
stage_order = ['P1', 'P3', 'P8', 'P14']
crosstab = crosstab[stage_order]
existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab.index]
crosstab = crosstab.loc[existing_mice]
print(crosstab)

# Print additional info about labels
print("\nLabel distribution (onnest_label):")
print(pd.Series(trimmed_dict['onnest_label']).value_counts().sort_index())

print("\n" + "="*80)
print("PROCESSING COMPLETE!")
print("="*80)
print(f"\nOriginal data: {len(full_dict['mouse_id'])} samples")
print(f"Trimmed data: {len(trimmed_dict['mouse_id'])} samples")
print(f"Reduction: {len(full_dict['mouse_id']) - len(trimmed_dict['mouse_id'])} samples removed")

Total samples in original data: 165210
Unique mice in original data: 14

PROCESSING ON-NEST DATA TRIMMING

After filtering to keep only specified mice:
  Total samples: 165210
  Unique mice: 14

Applying trim criteria:
  P1: first 4800 samples per mouse
  P14: first 1200 samples per mouse
  P3: first 3600 samples per mouse
  P8: first 2400 samples per mouse

After trimming:
  Total samples: 152400

✓ Saved trimmed on-nest data to: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_8Yes/combined/full_onnest_spec_features_8roi_Jan212026_Trim.pkl

Summary statistics after trimming:
period      P1    P3    P8   P14
mouse_id                        
C1_ELS32  4800  3600  2400  1200
C2_ELS18  4800  3600  2400  1200
C5_ELS40  4800  3600  2400  1200
C5_ELS20  4800  3600  2400  1200
C6_ELS42  4800  3600  2400  1200
C7_ELS11     0  3600  2400  1200
C7_ELS22  4800  3600  2400  1200
C7_ELS45  4800  3600  2400  1200
E1_ELS33  4800  3600  2400  1200
E2_ELS3      0  3600  2400  1

In [30]:
all_fns = get_filenames("/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi", "pkl")

filtered_files = all_fns

108 files in /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi


# 12. Jan 22 2026 — redo 1 Hz frequency band → generates `Spec_Features_1Hz_8roi/` pkls

In [31]:
# Initialize data structure to hold all combined data
all_data = {
    'power': [],
    'coh_sq_coherence': [],
    'freq_band': None,
    'region': None,
    'region_pair': None,
    'mouse_id': [],
    'period': []
}

# Track files that fail to load
failed_files = []

# Load and combine data from each file
print("Loading and combining feature files...")
for i, file_path in enumerate(filtered_files):
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
            
            # Append array data
            all_data['power'].append(data['power'])
            all_data['coh_sq_coherence'].append(data['coh_sq_coherence'])
            all_data['mouse_id'].append(data['mouse_id'])
            all_data['period'].append(data['period'])
            
            # Store static data (should be the same across all files)
            if all_data['freq_band'] is None:
                all_data['freq_band'] = data['freq_band']
            if all_data['region'] is None:
                all_data['region'] = data['region']
            if all_data['region_pair'] is None:
                all_data['region_pair'] = data['region_pair']
                
        if (i+1) % 20 == 0:
            print(f"  Processed {i+1}/{len(filtered_files)} files")
            
    except Exception as e:
        print(f"  ERROR loading file {file_path}: {str(e)}")
        failed_files.append((file_path, str(e)))

# Concatenate all the array data
print("\nConcatenating arrays...")
all_data['power'] = np.concatenate(all_data['power'], axis=0)
all_data['coh_sq_coherence'] = np.concatenate(all_data['coh_sq_coherence'], axis=0)
all_data['mouse_id'] = np.concatenate(all_data['mouse_id'], axis=0)
all_data['period'] = np.concatenate(all_data['period'], axis=0)

# Save the combined data
output_repo = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined"
os.makedirs(output_repo, exist_ok=True)  # os.path.dirname()
output_path = os.path.join(output_repo, "full_spec_features.pkl")
print(f"Saving combined data to {output_path}...")
with open(output_path, 'wb') as file:
    pickle.dump(all_data, file)

# Print summary
print("\n" + "-" * 80)
print(f"COMBINATION COMPLETE")
print(f"  Total files processed: {len(filtered_files) - len(failed_files)}/{len(filtered_files)}")
print(f"  Failed files: {len(failed_files)}")
print(f"  Output file: {output_path}")
print(f"  Data shape summary:")
print(f"    - power: {all_data['power'].shape}")
print(f"    - coh_sq_coherence: {all_data['coh_sq_coherence'].shape}")
print(f"    - mouse_id: {all_data['mouse_id'].shape}")
if 'period' in all_data and len(all_data['period']) > 0:
    print(f"    - period: {all_data['period'].shape}")
print("-" * 80)

if failed_files:
    print("\nList of failed files:")
    for i, (file, error) in enumerate(failed_files, 1):
        print(f"  {i}. {file} - Error: {error}")


Loading and combining feature files...
  Processed 20/108 files
  Processed 40/108 files
  Processed 60/108 files
  Processed 80/108 files
  Processed 100/108 files

Concatenating arrays...
Saving combined data to /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features.pkl...

--------------------------------------------------------------------------------
COMBINATION COMPLETE
  Total files processed: 108/108
  Failed files: 0
  Output file: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features.pkl
  Data shape summary:
    - power: (190145, 432)
    - coh_sq_coherence: (190145, 1512)
    - mouse_id: (190145,)
    - period: (190145,)
--------------------------------------------------------------------------------


In [3]:
TRAINING_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features.pkl"

X_FEATURE_LIST = ["power", "coh_sq_coherence"]
X_FEATURE_WEIGHTS = [1, 1]
Y_FEATURE = "period"

with open(TRAINING_DATA_FILE, "rb") as f:
    train_dict = pickle.load(f)

unique_periods = np.unique(train_dict[Y_FEATURE])
print("unique periods:")
print(unique_periods)
print("unique mouse id:")
print(np.unique(train_dict['mouse_id']))


full_dict = train_dict
print("Dictionary keys:")
for key in full_dict.keys():
    print(key)

# Print some sample elements
print("\nSample elements:")
for key in full_dict.keys():
    value = full_dict[key]
    print(f"\nKey: {key}")
    
    # Handle different types of data
    if isinstance(value, list) and len(value) > 0:
        print(f"Type: list, Length: {len(value)}")
        print(f"First 3 elements: {value[:3]}")
    elif isinstance(value, dict):
        print(f"Type: dict, Number of keys: {len(value)}")
        print(f"Dictionary keys: {list(value.keys())[:5]}")
    elif hasattr(value, 'shape'):  # For numpy arrays
        print(f"Type: {type(value)}")
        print(f"Shape: {value.shape}")
        print(f"Data type: {value.dtype}")
        # Print a small slice of the array
        if value.size > 0:
            print("Sample (first few elements):")
            if value.ndim == 1:
                print(value[:5])  # First 5 elements for 1D array
            else:
                print(value[:2, :2])  # First 2×2 elements for 2D+ array
    elif isinstance(value, (int, float, str, bool)):
        print(f"Value: {value}")
    else:
        print(f"Type: {type(value)}")

unique periods:
['Ges' 'P1' 'P14' 'P20' 'P3' 'P4 home' 'P8' 'Pre home']
unique mouse id:
['MouseC1ELS1' 'MouseC1F3ELS32' 'MouseC2F4ELS18' 'MouseC5F3ELS40'
 'MouseC5F4ELS20' 'MouseC6ELS9' 'MouseC6F5ELS42' 'MouseC7ELS11'
 'MouseC7F1ELS22' 'MouseC7F2ELS45' 'MouseE1F4ELS33' 'MouseE2ELS3'
 'MouseE2F2ELS35' 'MouseE3F1ELS37' 'MouseE4F2ELS39' 'MouseE5F1ELS41'
 'MouseE6F1ELS43' 'MouseE6F4ELS44']
Dictionary keys:
power
coh_sq_coherence
freq_band
region
region_pair
mouse_id
period

Sample elements:

Key: power
Type: <class 'numpy.ndarray'>
Shape: (190145, 432)
Data type: float64
Sample (first few elements):
[[0.49663639 0.78589876]
 [0.58879288 0.64505355]]

Key: coh_sq_coherence
Type: <class 'numpy.ndarray'>
Shape: (190145, 1512)
Data type: float64
Sample (first few elements):
[[0.92327289 0.99912536]
 [0.97414449 0.99091897]]

Key: freq_band
Type: list, Length: 54
First 3 elements: [(2, 3), (3, 4), (4, 5)]

Key: region
Type: list, Length: 432
First 3 elements: ['BLA', 'CeA', 'IL']

Key: region_

In [4]:
# Define the mice IDs to keep (with F notation)
c_mice_ids = ['C6ELS9', 'C7ELS11', 'C2F4ELS18', 'C5F4ELS20', 'C7F1ELS22', 
              'C1F3ELS32', 'C5F3ELS40', 'C6F5ELS42', 'C7F2ELS45']
e_mice_ids = ['E2ELS3', 'E1F4ELS33', 'E3F1ELS37', 'E4F2ELS39', 'E5F1ELS41', 'E6F4ELS44']

# Combine the mice IDs
mice_to_keep = c_mice_ids + e_mice_ids

# Create DataFrame
df = pd.DataFrame({
    'mouse_id': full_dict['mouse_id'],
    'period': full_dict['period']
})

# Extract the short mouse ID format (remove 'Mouse' prefix)
df['mouse_id_short'] = df['mouse_id'].str.replace('Mouse', '')

# Filter to keep only specified mice
df_filtered = df[df['mouse_id_short'].isin(mice_to_keep)]

# Define the desired order for periods/stages
stage_order = ['Pre home', 'Ges', 'P1', 'P3', 'P4 home', 'P8', 'P14', 'P20']

# Create crosstab
crosstab = pd.crosstab(df_filtered['mouse_id_short'], df_filtered['period'])

# Reorder columns according to stage_order (only include stages that exist in data)
existing_stages = [stage for stage in stage_order if stage in crosstab.columns]
crosstab = crosstab[existing_stages]

# Reorder rows to match the order in c_mice_ids and e_mice_ids
existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab.index]
crosstab = crosstab.loc[existing_mice]

print("Sample counts for each mouse_id and stage combination:")
print("=" * 100)
print(crosstab)

Sample counts for each mouse_id and stage combination:
period          Pre home  Ges    P1    P3  P4 home    P8   P14   P20
mouse_id_short                                                      
C6ELS9               203  212     0     0        0     0     0   213
C7ELS11              204  259     0  3604        0  3356  1304   212
C2F4ELS18            208  216  4908  3642        0  3178  1278   205
C5F4ELS20            213  234  5009  3692        0  2524  1311   207
C7F1ELS22            210  235  4862  4801        0  2489  1354   221
C1F3ELS32            404  411  4826  3756      759  2821  1258  1561
C5F3ELS40            402  445  5110  3625      600  2763  1562   486
C6F5ELS42            410  443  5289  4308      765  2610  1280   409
C7F2ELS45            403  482  4838  3695      819  2443  1797   414
E2ELS3               267  223     0  3785        0  2504  1213   207
E1F4ELS33            402  457  4982  3615      756  2686  1254   417
E3F1ELS37            402  436     0  3816      8

In [5]:
import pickle
import numpy as np
import pandas as pd

# Load the data
TRAINING_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features.pkl"

with open(TRAINING_DATA_FILE, "rb") as f:
    full_dict = pickle.load(f)

# Define the mice IDs to keep
c_mice_ids = ['C6ELS9', 'C7ELS11', 'C2F4ELS18', 'C5F4ELS20', 'C7F1ELS22', 
              'C1F3ELS32', 'C5F3ELS40', 'C6F5ELS42', 'C7F2ELS45']
e_mice_ids = ['E2ELS3', 'E1F4ELS33', 'E3F1ELS37', 'E4F2ELS39', 'E5F1ELS41', 'E6F4ELS44']
mice_to_keep = c_mice_ids + e_mice_ids

print(f"Total samples in original data: {len(full_dict['mouse_id'])}")
print(f"Unique mice in original data: {len(np.unique(full_dict['mouse_id']))}")

def filter_and_trim_data(full_dict, mice_to_keep, trim_criteria):
    """
    First filter to keep only specified mice, then trim data based on criteria.
    
    Parameters:
    - full_dict: dictionary containing all data
    - mice_to_keep: list of short mouse IDs to keep (without 'Mouse' prefix)
    - trim_criteria: dictionary mapping stage -> max_samples (or None to keep all)
    
    Returns:
    - trimmed_dict: dictionary with filtered and trimmed data
    """
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame({
        'mouse_id': full_dict['mouse_id'],
        'period': full_dict['period'],
        'index': np.arange(len(full_dict['mouse_id']))
    })
    
    # Add short mouse ID
    df['mouse_id_short'] = df['mouse_id'].str.replace('Mouse', '')
    
    # STEP 1: Filter to keep only specified mice
    df = df[df['mouse_id_short'].isin(mice_to_keep)].copy()
    
    print(f"\nAfter filtering to keep only specified mice:")
    print(f"  Total samples: {len(df)}")
    print(f"  Unique mice: {df['mouse_id_short'].nunique()}")
    
    # STEP 2: Apply trim criteria
    print(f"\nApplying trim criteria:")
    for stage in sorted(trim_criteria.keys()):
        max_samples = trim_criteria[stage]
        if max_samples is not None:
            print(f"  {stage}: first {max_samples} samples per mouse")
        else:
            print(f"  {stage}: keep all samples")
    
    # Group by mouse and period, then take first N samples for each group
    indices_to_keep = []
    
    for (mouse_id, period), group in df.groupby(['mouse_id', 'period']):
        max_samples = trim_criteria.get(period, None)
        
        if max_samples is not None:
            # Take first N samples
            selected_indices = group['index'].values[:max_samples]
        else:
            # Keep all samples
            selected_indices = group['index'].values
        
        indices_to_keep.extend(selected_indices)
    
    indices_to_keep = np.array(sorted(indices_to_keep))
    
    print(f"\nAfter trimming:")
    print(f"  Total samples: {len(indices_to_keep)}")
    
    # Create trimmed dictionary
    trimmed_dict = {}
    for key in full_dict.keys():
        if key in ['freq_band', 'region', 'region_pair']:
            # These are metadata, keep as is
            trimmed_dict[key] = full_dict[key]
        else:
            # These are arrays that need to be indexed
            trimmed_dict[key] = full_dict[key][indices_to_keep]
    
    return trimmed_dict

# Calculate samples based on time windows (3 seconds per sample)
samples_per_minute = 20  # 60 seconds / 3 seconds per sample
samples_per_hour = samples_per_minute * 60  # 1200 samples per hour

# # ============================================================================
# # TRIM CRITERIA 1: Post stages only
# # ============================================================================
# print("\n" + "="*80)
# print("PROCESSING TRIM CRITERIA 1: Post stages only")
# print("="*80)

# trim_criteria_1 = {
#     'P1': 4 * samples_per_hour,      # 4 hours = 4800 samples
#     'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
#     'P8': 2 * samples_per_hour,      # 2 hours = 2400 samples
#     'P14': 1 * samples_per_hour,     # 1 hour = 1200 samples
#     'P4 home': 20 * samples_per_minute,  # 20 mins = 400 samples
#     # Other stages keep all samples
#     'Pre home': None,
#     'Ges': None,
#     'P20': None
# }

# trimmed_dict_1 = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria_1)

# # Save Trim Criteria 1
# output_file_1 = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features_Trim_Post.pkl"
# with open(output_file_1, "wb") as f:
#     pickle.dump(trimmed_dict_1, f)
# print(f"\n✓ Saved Trim Criteria 1 to: {output_file_1}")

# # Print summary for Trim Criteria 1
# print("\nSummary statistics for Trim Criteria 1:")
# df1 = pd.DataFrame({
#     'mouse_id_short': [mid.replace('Mouse', '') for mid in trimmed_dict_1['mouse_id']],
#     'period': trimmed_dict_1['period']
# })
# crosstab1 = pd.crosstab(df1['mouse_id_short'], df1['period'])
# stage_order = ['Pre home', 'Ges', 'P1', 'P3', 'P4 home', 'P8', 'P14', 'P20']
# existing_stages = [stage for stage in stage_order if stage in crosstab1.columns]
# crosstab1 = crosstab1[existing_stages]
# existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab1.index]
# crosstab1 = crosstab1.loc[existing_mice]
# print(crosstab1)

# ============================================================================
# TRIM CRITERIA 2: All stages with specific limits
# ============================================================================
print("\n" + "="*80)
print("PROCESSING TRIM CRITERIA 2: All stages with limits")
print("="*80)

trim_criteria_2 = {
    'P1': 4 * samples_per_hour,      # 4 hours = 4800 samples
    'P3': 3 * samples_per_hour,      # 3 hours = 3600 samples
    'P8': 2 * samples_per_hour,      # 2 hours = 2400 samples
    'P14': 1 * samples_per_hour,     # 1 hour = 1200 samples
    'P4 home': 20 * samples_per_minute,  # 20 mins = 400 samples
    'Pre home': 10 * samples_per_minute,  # 10 mins = 200 samples
    'Ges': 10 * samples_per_minute,  # 10 mins = 200 samples
    'P20': 10 * samples_per_minute   # 10 mins = 200 samples
}

trimmed_dict_2 = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria_2)

# Save Trim Criteria 2
output_file_2 = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features_Trim_All.pkl"
with open(output_file_2, "wb") as f:
    pickle.dump(trimmed_dict_2, f)
print(f"\n✓ Saved Trim Criteria 2 to: {output_file_2}")

# Print summary for Trim Criteria 2
print("\nSummary statistics for Trim Criteria 2:")
df2 = pd.DataFrame({
    'mouse_id_short': [mid.replace('Mouse', '') for mid in trimmed_dict_2['mouse_id']],
    'period': trimmed_dict_2['period']
})
crosstab2 = pd.crosstab(df2['mouse_id_short'], df2['period'])
crosstab2 = crosstab2[existing_stages]
existing_mice_2 = [mouse for mouse in mice_to_keep if mouse in crosstab2.index]
crosstab2 = crosstab2.loc[existing_mice_2]
print(crosstab2)

print("\n" + "="*80)
print("PROCESSING COMPLETE!")
print("="*80)
print(f"\nOriginal data: {len(full_dict['mouse_id'])} samples")
print(f"Trim Criteria 1: {len(trimmed_dict_1['mouse_id'])} samples")
print(f"Trim Criteria 2: {len(trimmed_dict_2['mouse_id'])} samples")

Total samples in original data: 190145
Unique mice in original data: 18

PROCESSING TRIM CRITERIA 2: All stages with limits

After filtering to keep only specified mice:
  Total samples: 189123
  Unique mice: 15

Applying trim criteria:
  Ges: first 200 samples per mouse
  P1: first 4800 samples per mouse
  P14: first 1200 samples per mouse
  P20: first 200 samples per mouse
  P3: first 3600 samples per mouse
  P4 home: first 400 samples per mouse
  P8: first 2400 samples per mouse
  Pre home: first 200 samples per mouse

After trimming:
  Total samples: 164800

✓ Saved Trim Criteria 2 to: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_spec_features_Trim_All.pkl

Summary statistics for Trim Criteria 2:
period          Pre home  Ges    P1    P3  P4 home    P8   P14  P20
mouse_id_short                                                     
C6ELS9               200  200     0     0        0     0     0  200
C7ELS11              200  200     0

NameError: name 'trimmed_dict_1' is not defined

In [50]:
import pickle
import numpy as np
import pandas as pd

# Load the data
FULL_DATA_FILE = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_onnest_spec_features.pkl"

with open(FULL_DATA_FILE, 'rb') as f:
    full_dict = pickle.load(f)

# Define the mice IDs to keep (matching the actual format in data)
c_mice_ids = ['MouseC1F3ELS32', 'MouseC2F4ELS18', 'MouseC5F3ELS40', 'MouseC5F4ELS20', 
              'MouseC6F5ELS42', 'MouseC7ELS11', 'MouseC7F1ELS22', 'MouseC7F2ELS45']
e_mice_ids = ['MouseE1F4ELS33', 'MouseE2ELS3', 'MouseE3F1ELS37', 'MouseE4F2ELS39', 
              'MouseE5F1ELS41', 'MouseE6F4ELS44']
mice_to_keep = c_mice_ids + e_mice_ids

print(f"Total samples in original data: {len(full_dict['mouse_id'])}")
print(f"Unique mice in original data: {len(np.unique(full_dict['mouse_id']))}")

def filter_and_trim_data(full_dict, mice_to_keep, trim_criteria):
    """
    First filter to keep only specified mice, then trim data based on criteria.
    
    Parameters:
    - full_dict: dictionary containing all data
    - mice_to_keep: list of mouse IDs to keep (exact format as in data)
    - trim_criteria: dictionary mapping stage -> max_samples
    
    Returns:
    - trimmed_dict: dictionary with filtered and trimmed data
    """
    # Convert to DataFrame for easier manipulation
    df = pd.DataFrame({
        'mouse_id': full_dict['mouse_id'],
        'period': full_dict['period'],
        'index': np.arange(len(full_dict['mouse_id']))
    })
    
    # STEP 1: Filter to keep only specified mice
    df = df[df['mouse_id'].isin(mice_to_keep)].copy()
    
    print(f"\nAfter filtering to keep only specified mice:")
    print(f"  Total samples: {len(df)}")
    print(f"  Unique mice: {df['mouse_id'].nunique()}")
    
    if len(df) == 0:
        print("\n*** WARNING: No samples found matching the specified mice! ***")
        print("Please check the mouse_id format in your data.")
        return None
    
    # STEP 2: Apply trim criteria
    print(f"\nApplying trim criteria:")
    for stage in sorted(trim_criteria.keys()):
        max_samples = trim_criteria[stage]
        print(f"  {stage}: first {max_samples} samples per mouse")
    
    # Group by mouse and period, then take first N samples for each group
    indices_to_keep = []
    
    for (mouse_id, period), group in df.groupby(['mouse_id', 'period']):
        max_samples = trim_criteria.get(period)
        
        if max_samples is not None:
            # Take first N samples
            selected_indices = group['index'].values[:max_samples]
        else:
            # This shouldn't happen, but keep all if no criteria specified
            selected_indices = group['index'].values
        
        indices_to_keep.extend(selected_indices)
    
    # Convert to integer array and sort
    indices_to_keep = np.array(sorted(indices_to_keep), dtype=np.int64)
    
    print(f"\nAfter trimming:")
    print(f"  Total samples: {len(indices_to_keep)}")
    
    # Create trimmed dictionary
    trimmed_dict = {}
    for key in full_dict.keys():
        if key in ['freq_band', 'region', 'region_pair']:
            # These are metadata, keep as is
            trimmed_dict[key] = full_dict[key]
        else:
            # These are arrays that need to be indexed
            trimmed_dict[key] = full_dict[key][indices_to_keep]
    
    return trimmed_dict

# Calculate samples based on time windows (3 seconds per sample)
samples_per_minute = 20  # 60 seconds / 3 seconds per sample
samples_per_hour = samples_per_minute * 60  # 1200 samples per hour

# ============================================================================
# TRIM ON-NEST DATA: P1/3/8/14 only
# ============================================================================
print("\n" + "="*80)
print("PROCESSING ON-NEST DATA TRIMMING")
print("="*80)

trim_criteria = {
    'P1': 4 * samples_per_hour,   # 4 hours = 4800 samples
    'P3': 3 * samples_per_hour,   # 3 hours = 3600 samples
    'P8': 2 * samples_per_hour,   # 2 hours = 2400 samples
    'P14': 1 * samples_per_hour,  # 1 hour = 1200 samples    
}

trimmed_dict = filter_and_trim_data(full_dict, mice_to_keep, trim_criteria)

if trimmed_dict is not None:
    # Save trimmed data
    output_file = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_onnest_spec_features_Trim.pkl"

    with open(output_file, "wb") as f:
        pickle.dump(trimmed_dict, f)
    print(f"\n✓ Saved trimmed on-nest data to: {output_file}")

    # Print summary statistics
    print("\nSummary statistics after trimming:")
    df_trimmed = pd.DataFrame({
        'mouse_id': trimmed_dict['mouse_id'],
        'period': trimmed_dict['period']
    })
    crosstab = pd.crosstab(df_trimmed['mouse_id'], df_trimmed['period'])
    stage_order = ['P1', 'P3', 'P8', 'P14']
    crosstab = crosstab[[col for col in stage_order if col in crosstab.columns]]
    existing_mice = [mouse for mouse in mice_to_keep if mouse in crosstab.index]
    crosstab = crosstab.loc[existing_mice]
    print(crosstab)

    # Print additional info about labels
    if 'onnest_label' in trimmed_dict:
        print("\nLabel distribution (onnest_label):")
        print(pd.Series(trimmed_dict['onnest_label']).value_counts().sort_index())

    print("\n" + "="*80)
    print("PROCESSING COMPLETE!")
    print("="*80)
    print(f"\nOriginal data: {len(full_dict['mouse_id'])} samples")
    print(f"Trimmed data: {len(trimmed_dict['mouse_id'])} samples")
    print(f"Reduction: {len(full_dict['mouse_id']) - len(trimmed_dict['mouse_id'])} samples removed")
else:
    print("\n*** FAILED: No data to save ***")

Total samples in original data: 165210
Unique mice in original data: 14

PROCESSING ON-NEST DATA TRIMMING

After filtering to keep only specified mice:
  Total samples: 165210
  Unique mice: 14

Applying trim criteria:
  P1: first 4800 samples per mouse
  P14: first 1200 samples per mouse
  P3: first 3600 samples per mouse
  P8: first 2400 samples per mouse

After trimming:
  Total samples: 152400

✓ Saved trimmed on-nest data to: /Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior/Spec_Features_1Hz_8roi/combined/full_onnest_spec_features_Trim.pkl

Summary statistics after trimming:
period            P1    P3    P8   P14
mouse_id                              
MouseC1F3ELS32  4800  3600  2400  1200
MouseC2F4ELS18  4800  3600  2400  1200
MouseC5F3ELS40  4800  3600  2400  1200
MouseC5F4ELS20  4800  3600  2400  1200
MouseC6F5ELS42  4800  3600  2400  1200
MouseC7ELS11       0  3600  2400  1200
MouseC7F1ELS22  4800  3600  2400  1200
MouseC7F2ELS45  4800  3600  2400  1200
MouseE1F4ELS3